# Trading Data and ML Foundations

Maps to `design3.md` Phase 4.

This notebook is the bridge from generic platform engineering into your chosen vertical: **trading AI/ML research data platform**. Every concept here is something that separates people who build trading systems from people who build generic CRUD apps. The stakes are real: a single leakage bug in a backtesting pipeline can make a worthless strategy look profitable, and a single symbology mismatch can route an order to the wrong instrument.

**What this notebook covers:**

1. Market data entities (Instrument, Trade, Quote, Bar, CorporateAction) with full dataclass implementations
2. Symbology and canonical identifiers — why every vendor disagrees and how to unify them
3. Venue calendars and session boundaries — when markets are open and why it matters
4. Time concepts: event time, processing time, as-of time
5. Point-in-time correctness — the single most important concept for ML on trading data
6. Leakage prevention — how future data sneaks into your training set
7. Feature engineering with explicit lookback windows
8. Dataset versioning and snapshots
9. FeatureView: declarative, reproducible feature specifications
10. Experiment tracking and model artifact traceability
11. Data quality reporting
12. Vendor mapping and reconciliation exercise
13. Mini lab: end-to-end trading data pipeline

**Audience assumptions:** You have 4+ years as a data/platform engineer with 3 years on an institutional FX trading desk. You know what a VWAP is. You know what Kafka does. This notebook does not explain trading from scratch — it formalizes what you already know into reproducible, auditable data structures.

**Working rule:** Read the explanation, run the code, modify the examples, write one short note in your own words before moving on.

## 1. Market Data Entities

Every trading data platform needs a small set of canonical data models. These are not application-specific — they are the lingua franca that every downstream consumer (analytics, ML, risk, compliance) speaks. Getting these wrong poisons everything downstream.

The five core entities:

| Entity | What it represents | Update frequency |
|--------|-------------------|-----------------|
| **Instrument** | Canonical metadata about a tradeable asset | Rarely (days/months) |
| **Trade** | An executed market transaction | Millions per day |
| **Quote** | Best bid/offer (top-of-book) | Tens of millions per day |
| **Bar** | Aggregated OHLCV over a time interval | Once per interval |
| **CorporateAction** | Events that affect price/volume comparability | Occasionally |

Design principles that apply to all five:

- **Decimal for prices.** Floating-point arithmetic introduces rounding errors that compound across millions of records. In FX, a pip is 0.0001 — you cannot afford `0.1 + 0.2 = 0.30000000000000004`. Python's `decimal.Decimal` gives exact decimal arithmetic.
- **Timezone-aware datetimes.** Always store timestamps in UTC. A naive datetime is a bug waiting to happen when you cross venue boundaries.
- **Frozen dataclasses.** Market data records should be immutable after creation. If you need to "correct" a record, you create a new record with a new transaction timestamp — the original stays in the audit trail.

### 1.1 Instrument

An Instrument is canonical metadata about a tradeable asset. It answers the question: "What is this thing, and how do I trade it?"

**Why effective_start and effective_end?** Instruments are not static. Ticker symbols get reassigned (META was formerly FB). Companies delist. Mergers create new instruments. Corporate actions change lot sizes. This is Slowly Changing Dimension Type 2 (SCD2) — instead of overwriting the old record, you close it with an `effective_end` date and open a new record with the updated fields. This preserves history: if you ask "what was AAPL's tick size on 2020-01-15?", you can answer it.

**The ticker symbol trap.** Never use ticker symbols as stable identifiers. Bloomberg uses `AAPL US Equity`, Reuters uses `AAPL.O`, the exchange uses `AAPL`, and next year the company might change its name. Use a stable internal ID and map vendor symbols to it (more on this in section 2).

**Industry canonical IDs:**
- **ISIN** (International Securities Identification Number): 12-character alphanumeric, globally unique per issue. Example: `US0378331005` for Apple.
- **FIGI** (Financial Instrument Global Identifier): Bloomberg's open standard, 12 characters. Example: `BBG000B9XRY4`.
- **CUSIP**: 9-character, North American focus. Example: `037833100` for Apple.
- **SEDOL**: 7-character, primarily UK/Irish securities.

In FX, the situation is simpler — currency pairs are identified by ISO 4217 codes (EURUSD, USDJPY) — but even here, quoting conventions differ between venues.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from decimal import Decimal
from enum import Enum
from typing import Optional
import uuid
import hashlib
import json


class AssetClass(Enum):
    FX = "FX"
    EQUITY = "EQUITY"
    FIXED_INCOME = "FIXED_INCOME"
    COMMODITY = "COMMODITY"
    CRYPTO = "CRYPTO"


@dataclass(frozen=True)
class Instrument:
    """Canonical metadata about a tradeable asset (SCD Type 2)."""

    instrument_id: str  # stable internal ID, never changes
    symbol: str  # human-readable, but NOT a stable key
    venue: str  # where this instrument trades
    asset_class: AssetClass
    currency: str  # ISO 4217 quote currency
    tick_size: Decimal  # minimum price increment
    lot_size: Decimal  # minimum tradeable quantity
    effective_start: datetime  # when this version became active (UTC)
    effective_end: Optional[datetime] = None  # None = currently active
    isin: Optional[str] = None  # International Securities ID
    figi: Optional[str] = None  # Bloomberg FIGI


# Example: Apple equity on NYSE
aapl = Instrument(
    instrument_id="INT-000001",
    symbol="AAPL",
    venue="NYSE",
    asset_class=AssetClass.EQUITY,
    currency="USD",
    tick_size=Decimal("0.01"),
    lot_size=Decimal("1"),
    effective_start=datetime(2020, 1, 1, tzinfo=timezone.utc),
    isin="US0378331005",
    figi="BBG000B9XRY4",
)

# Example: EURUSD spot FX
eurusd = Instrument(
    instrument_id="INT-FX-0001",
    symbol="EURUSD",
    venue="EBS",
    asset_class=AssetClass.FX,
    currency="USD",
    tick_size=Decimal("0.00001"),  # half-pip on EBS
    lot_size=Decimal("1000000"),  # standard lot = 1M base currency
    effective_start=datetime(2020, 1, 1, tzinfo=timezone.utc),
)

# SCD2 in action: Meta was formerly Facebook
fb_old = Instrument(
    instrument_id="INT-000042",
    symbol="FB",
    venue="NASDAQ",
    asset_class=AssetClass.EQUITY,
    currency="USD",
    tick_size=Decimal("0.01"),
    lot_size=Decimal("1"),
    effective_start=datetime(2012, 5, 18, tzinfo=timezone.utc),
    effective_end=datetime(2022, 6, 9, tzinfo=timezone.utc),  # closed
)

meta_new = Instrument(
    instrument_id="INT-000042",  # same internal ID
    symbol="META",  # new ticker
    venue="NASDAQ",
    asset_class=AssetClass.EQUITY,
    currency="USD",
    tick_size=Decimal("0.01"),
    lot_size=Decimal("1"),
    effective_start=datetime(2022, 6, 9, tzinfo=timezone.utc),  # new version
)

print(f"AAPL: {aapl.instrument_id}, tick={aapl.tick_size}, ISIN={aapl.isin}")
print(f"EURUSD: {eurusd.instrument_id}, tick={eurusd.tick_size}, lot={eurusd.lot_size:,}")
print(f"FB closed: {fb_old.effective_end}")
print(f"META opened: {meta_new.effective_start}")
print(f"Same entity? {fb_old.instrument_id == meta_new.instrument_id}")

# Try next:
# 1. Add a GBPUSD instrument with proper tick size for your venue.
# 2. Try to mutate aapl.symbol — observe the frozen dataclass error.
# 3. Query: "what was the symbol for INT-000042 on 2021-01-01?"

### 1.2 Trade

A Trade is an executed market transaction — someone bought, someone sold, a price was agreed, a quantity changed hands. Trades are the ground truth of market activity. Everything else (bars, VWAP, analytics) is derived from trades.

**Why Decimal for price?** Consider computing the total value of 1,000 trades. With `float`, rounding errors accumulate silently. In FX where you might be dealing with notional values in the hundreds of millions, even small errors compound into real money. `Decimal("1.08450")` is exact. `float(1.08450)` is `1.0844999999999998`.

**Why both event_time and ingest_time?** In a perfect world, these would be identical. In reality:
- A trade happens on the exchange at 10:00:00.123 (event_time)
- Your Kafka consumer picks it up at 10:00:00.456 (ingest_time)
- In batch mode, the trade might not arrive until 10:05:00 or even the next day

This divergence matters for everything: windowed aggregations, late-arrival handling, point-in-time correctness. If you only store one timestamp, you cannot distinguish "trade happened late" from "trade happened on time."

**Trade conditions:** Exchanges flag special trade types — block trades (large negotiated trades), odd lots (below standard lot size), crosses (buyer and seller are the same entity), and more. These flags affect whether the trade should be included in official VWAP calculations, whether it updates the last-sale price, and whether it triggers circuit breakers. Ignoring conditions is a common source of bad analytics.

In [ ]:
class Side(Enum):
    BUY = "BUY"
    SELL = "SELL"


class TradeCondition(Enum):
    REGULAR = "REGULAR"
    BLOCK = "BLOCK"  # large negotiated trade
    ODD_LOT = "ODD_LOT"  # below standard lot size
    CROSS = "CROSS"  # same entity on both sides
    LATE_REPORT = "LATE_REPORT"  # reported after the fact


@dataclass(frozen=True)
class Trade:
    """An executed market transaction."""

    source: str  # originating vendor/exchange
    trade_id: str  # unique within source
    instrument_id: str  # FK to Instrument
    event_time: datetime  # when the trade actually happened (UTC)
    ingest_time: datetime  # when our system received it (UTC)
    price: Decimal  # execution price — ALWAYS Decimal
    size: Decimal  # quantity traded
    side: Side  # buyer-initiated or seller-initiated
    conditions: tuple[TradeCondition, ...] = (TradeCondition.REGULAR,)


# Demonstrate why Decimal matters
print("=== Float vs Decimal ===")
print(f"float:   0.1 + 0.2 = {0.1 + 0.2}")
print(f"Decimal: 0.1 + 0.2 = {Decimal('0.1') + Decimal('0.2')}")
print()

# Accumulate 10,000 trades at 1.08450 — float vs Decimal
float_total = sum(1.08450 for _ in range(10_000))
decimal_total = sum(Decimal("1.08450") for _ in range(10_000))
print(f"10,000 x 1.08450:")
print(f"  float sum:   {float_total}")
print(f"  Decimal sum: {decimal_total}")
print(f"  difference:  {abs(Decimal(str(float_total)) - decimal_total)}")
print()

now = datetime(2026, 4, 4, 8, 15, 0, 123_000, tzinfo=timezone.utc)
ingest = now + timedelta(milliseconds=333)

trade = Trade(
    source="EBS",
    trade_id="EBS-20260404-0000001",
    instrument_id="INT-FX-0001",
    event_time=now,
    ingest_time=ingest,
    price=Decimal("1.08450"),
    size=Decimal("5000000"),
    side=Side.BUY,
)

block_trade = Trade(
    source="BLOOMBERG",
    trade_id="BB-20260404-0042",
    instrument_id="INT-000001",
    event_time=now,
    ingest_time=ingest + timedelta(seconds=2),
    price=Decimal("178.52"),
    size=Decimal("500000"),
    side=Side.SELL,
    conditions=(TradeCondition.BLOCK,),
)

print(f"FX trade: {trade.instrument_id} {trade.side.value} {trade.size:,} @ {trade.price}")
print(f"  event_time:  {trade.event_time.isoformat()}")
print(f"  ingest_time: {trade.ingest_time.isoformat()}")
print(f"  latency:     {(trade.ingest_time - trade.event_time).total_seconds() * 1000:.0f}ms")
print()
print(f"Block trade: {block_trade.instrument_id} {block_trade.conditions[0].value}")
print(f"  {block_trade.size:,} shares @ ${block_trade.price}")

# Try next:
# 1. Create a late-reported trade where ingest_time is 5 minutes after event_time.
# 2. Compute notional value (price * size) for the FX trade using Decimal arithmetic.
# 3. Filter a list of trades to exclude BLOCK and ODD_LOT conditions.

### 1.3 Quote

A Quote represents the best bid and offer (BBO) at top-of-book — the best price someone is willing to buy at (bid) and the best price someone is willing to sell at (ask). Quotes are the heartbeat of market liquidity.

**Spread = ask - bid.** This is the most basic measure of liquidity. Tight spreads mean a liquid market where you can trade at low cost. Wide spreads mean illiquidity, risk, or low activity. In major FX pairs during London hours, the spread might be 0.1 pips. In an illiquid emerging-market currency at 3am, it might be 50 pips.

**Mid price = (bid + ask) / 2.** This is the "fair value" estimate used as a reference price in many analytics. It is not a price you can actually trade at — it is a theoretical construct.

**Why quotes matter for ML:** Quote data is far richer than trade data for modeling. The bid-ask spread, the sizes on each side, and how they change over time encode information about supply/demand imbalances, market maker behavior, and impending price moves. Many alpha signals come from quote dynamics, not trade dynamics.

In [ ]:
@dataclass(frozen=True)
class Quote:
    """Best bid and offer (top-of-book) at a point in time."""

    source: str
    instrument_id: str
    event_time: datetime  # when the quote was valid on the venue (UTC)
    ingest_time: datetime  # when our system received it (UTC)
    bid_price: Decimal
    bid_size: Decimal
    ask_price: Decimal
    ask_size: Decimal

    @property
    def spread(self) -> Decimal:
        """Spread = ask - bid. Basic liquidity indicator."""
        return self.ask_price - self.bid_price

    @property
    def mid_price(self) -> Decimal:
        """Mid = (bid + ask) / 2. Theoretical fair value."""
        return (self.bid_price + self.ask_price) / 2

    @property
    def spread_bps(self) -> Decimal:
        """Spread in basis points relative to mid price."""
        mid = self.mid_price
        if mid == 0:
            return Decimal("0")
        return (self.spread / mid) * 10_000


# Liquid FX quote during London session
liquid_quote = Quote(
    source="EBS",
    instrument_id="INT-FX-0001",
    event_time=datetime(2026, 4, 4, 10, 0, 0, tzinfo=timezone.utc),
    ingest_time=datetime(2026, 4, 4, 10, 0, 0, 150_000, tzinfo=timezone.utc),
    bid_price=Decimal("1.08445"),
    bid_size=Decimal("10000000"),
    ask_price=Decimal("1.08450"),
    ask_size=Decimal("8000000"),
)

# Illiquid quote during off-hours
illiquid_quote = Quote(
    source="EBS",
    instrument_id="INT-FX-0001",
    event_time=datetime(2026, 4, 5, 3, 0, 0, tzinfo=timezone.utc),
    ingest_time=datetime(2026, 4, 5, 3, 0, 0, 500_000, tzinfo=timezone.utc),
    bid_price=Decimal("1.08400"),
    bid_size=Decimal("1000000"),
    ask_price=Decimal("1.08500"),
    ask_size=Decimal("500000"),
)

for label, q in [("Liquid (London)", liquid_quote), ("Illiquid (3am)", illiquid_quote)]:
    print(f"{label}:")
    print(f"  bid={q.bid_price} ({q.bid_size:,})  ask={q.ask_price} ({q.ask_size:,})")
    print(f"  spread={q.spread}  mid={q.mid_price}  spread_bps={q.spread_bps:.2f}")
    print()

# Try next:
# 1. Create a quote for USDJPY with appropriate price levels.
# 2. Compare bid_size vs ask_size — which side has more resting liquidity?
# 3. Compute the "imbalance ratio": (bid_size - ask_size) / (bid_size + ask_size).

### 1.4 Bar (OHLCV)

A Bar is aggregated price data over a fixed time interval. The name comes from the visual representation on a bar chart — each bar shows the open, high, low, and close prices plus volume for one period.

**OHLCV fields:**
- **Open**: first trade price in the interval
- **High**: maximum trade price in the interval
- **Low**: minimum trade price in the interval
- **Close**: last trade price in the interval
- **Volume**: total quantity traded in the interval

**VWAP (Volume-Weighted Average Price)** = sum(price_i x volume_i) / sum(volume_i). This is the most important derived metric on a bar. VWAP represents the "average price paid" during the interval, weighted by how much was actually traded at each price. It is the standard benchmark for execution quality — if you bought at a price below VWAP, you did better than the average participant.

**Bar intervals:** 1-second, 1-minute, 5-minute, 15-minute, 1-hour, 1-day are standard. The choice depends on the use case. High-frequency strategies use sub-second bars. Swing trading uses daily bars. The key rule: bars are always computed from trades, never from other bars (resampling bars introduces errors because you lose intra-bar price information).

**Adjustment flag:** Corporate actions (splits, dividends) change the price level. A 2:1 stock split halves the price overnight — if you plot unadjusted bars, it looks like a 50% crash. The `adjusted` flag tells consumers whether corporate-action adjustments have been applied. More on this in section 1.5.

In [ ]:
@dataclass(frozen=True)
class Bar:
    """Aggregated OHLCV price data over a time interval."""

    instrument_id: str
    bar_start: datetime  # inclusive start of the interval (UTC)
    bar_end: datetime  # exclusive end of the interval (UTC)
    open: Decimal
    high: Decimal
    low: Decimal
    close: Decimal
    volume: Decimal
    vwap: Decimal  # volume-weighted average price
    trade_count: int  # number of trades in this bar
    adjusted: bool = False  # whether corporate actions have been applied


def compute_bar(instrument_id: str, trades: list[Trade], bar_start: datetime, bar_end: datetime) -> Bar:
    """Build a bar from a list of trades within the interval.

    This is how bars SHOULD be computed: from raw trades, not from other bars.
    """
    interval_trades = [
        t for t in trades
        if t.instrument_id == instrument_id and bar_start <= t.event_time < bar_end
    ]

    if not interval_trades:
        raise ValueError(f"No trades for {instrument_id} in [{bar_start}, {bar_end})")

    sorted_trades = sorted(interval_trades, key=lambda t: t.event_time)

    prices = [t.price for t in sorted_trades]
    total_volume = sum(t.size for t in sorted_trades)
    vwap_numerator = sum(t.price * t.size for t in sorted_trades)
    vwap = vwap_numerator / total_volume

    return Bar(
        instrument_id=instrument_id,
        bar_start=bar_start,
        bar_end=bar_end,
        open=sorted_trades[0].price,
        high=max(prices),
        low=min(prices),
        close=sorted_trades[-1].price,
        volume=total_volume,
        vwap=vwap,
        trade_count=len(sorted_trades),
    )


# Generate some sample trades for bar construction
sample_trades = []
base_time = datetime(2026, 4, 4, 10, 0, 0, tzinfo=timezone.utc)
trade_data = [
    (0, "1.08450", "2000000", Side.BUY),
    (5, "1.08455", "3000000", Side.BUY),
    (15, "1.08440", "1500000", Side.SELL),
    (30, "1.08460", "5000000", Side.BUY),
    (45, "1.08445", "2500000", Side.SELL),
]

for offset_sec, price, size, side in trade_data:
    t = Trade(
        source="EBS",
        trade_id=f"EBS-{offset_sec}",
        instrument_id="INT-FX-0001",
        event_time=base_time + timedelta(seconds=offset_sec),
        ingest_time=base_time + timedelta(seconds=offset_sec, milliseconds=200),
        price=Decimal(price),
        size=Decimal(size),
        side=side,
    )
    sample_trades.append(t)

bar = compute_bar(
    instrument_id="INT-FX-0001",
    trades=sample_trades,
    bar_start=base_time,
    bar_end=base_time + timedelta(minutes=1),
)

print(f"1-minute bar for {bar.instrument_id}:")
print(f"  interval: [{bar.bar_start.strftime('%H:%M:%S')}, {bar.bar_end.strftime('%H:%M:%S')})")
print(f"  O={bar.open} H={bar.high} L={bar.low} C={bar.close}")
print(f"  volume={bar.volume:,}  trades={bar.trade_count}")
print(f"  VWAP={bar.vwap:.5f}")
print()

# Verify VWAP manually
manual_vwap = sum(Decimal(p) * Decimal(s) for _, p, s, _ in trade_data) / sum(Decimal(s) for _, _, s, _ in trade_data)
print(f"  Manual VWAP check: {manual_vwap:.5f} (match={bar.vwap == manual_vwap})")

# Try next:
# 1. Add more trades and compute a 5-minute bar.
# 2. What happens if you try to compute a bar with zero trades?
# 3. Compute VWAP deviation: (close - vwap) / vwap — positive means close was above average.

### 1.5 CorporateAction

A CorporateAction is an event initiated by a company that affects the price or volume of its securities. The most common types:

- **Stock split** (e.g., 2:1): each existing share becomes 2 shares, price halves. A $200 stock becomes $100 overnight with twice the shares outstanding.
- **Reverse split** (e.g., 1:10): 10 shares become 1, price multiplies by 10. Often done to avoid delisting.
- **Cash dividend**: company pays cash per share. On the ex-date, the stock price theoretically drops by the dividend amount.
- **Stock dividend**: company issues additional shares instead of cash.
- **Merger/acquisition**: one instrument ceases to exist or converts to another.
- **Spin-off**: a division becomes a separate company with its own shares.

**Why this matters for data engineering:** If you store raw unadjusted prices and then compute a 20-day moving average across a 2:1 split, the pre-split prices are 2x the post-split prices. Your moving average will show a massive discontinuity that has nothing to do with actual market behavior. Every feature, every backtest, every model that uses historical prices must either:

1. Use **adjusted prices** (multiply all pre-split prices by 0.5), or
2. Use **returns** (percent changes) which are adjustment-invariant for splits

**Adjustment method:** For a split with ratio R:1, multiply all pre-split prices by 1/R and multiply all pre-split volumes by R. For dividends, the adjustment factor is (price - dividend) / price applied to all pre-dividend data.

In FX, corporate actions are not directly applicable to spot rates, but they matter for FX options, forwards, and any equity-linked FX products.

In [ ]:
class ActionType(Enum):
    SPLIT = "SPLIT"
    REVERSE_SPLIT = "REVERSE_SPLIT"
    CASH_DIVIDEND = "CASH_DIVIDEND"
    STOCK_DIVIDEND = "STOCK_DIVIDEND"
    MERGER = "MERGER"
    SPINOFF = "SPINOFF"


@dataclass(frozen=True)
class CorporateAction:
    """An event that affects price/volume comparability of historical data."""

    instrument_id: str
    action_type: ActionType
    effective_date: datetime  # when the action takes effect (UTC)
    ratio: Optional[Decimal] = None  # split ratio (e.g., 2.0 for 2:1 split)
    cash_amount: Optional[Decimal] = None  # dividend per share
    source: str = "EXCHANGE"


def adjust_bars_for_split(
    bars: list[Bar], action: CorporateAction
) -> list[Bar]:
    """Apply a split adjustment to historical bars.

    For a 2:1 split: pre-split prices are divided by 2, volumes multiplied by 2.
    This makes the price series continuous across the split date.
    """
    if action.action_type not in (ActionType.SPLIT, ActionType.REVERSE_SPLIT):
        raise ValueError(f"Expected split action, got {action.action_type}")
    if action.ratio is None:
        raise ValueError("Split action requires a ratio")

    adjusted = []
    for b in bars:
        if b.bar_start < action.effective_date:
            # Pre-split: adjust prices down, volumes up
            factor = Decimal("1") / action.ratio
            adjusted.append(Bar(
                instrument_id=b.instrument_id,
                bar_start=b.bar_start,
                bar_end=b.bar_end,
                open=b.open * factor,
                high=b.high * factor,
                low=b.low * factor,
                close=b.close * factor,
                volume=b.volume * action.ratio,
                vwap=b.vwap * factor,
                trade_count=b.trade_count,
                adjusted=True,
            ))
        else:
            # Post-split: already at new price level
            adjusted.append(Bar(
                instrument_id=b.instrument_id,
                bar_start=b.bar_start,
                bar_end=b.bar_end,
                open=b.open,
                high=b.high,
                low=b.low,
                close=b.close,
                volume=b.volume,
                vwap=b.vwap,
                trade_count=b.trade_count,
                adjusted=True,
            ))
    return adjusted


# Demonstrate: Apple 4:1 stock split (August 2020)
split = CorporateAction(
    instrument_id="INT-000001",
    action_type=ActionType.SPLIT,
    effective_date=datetime(2020, 8, 31, tzinfo=timezone.utc),
    ratio=Decimal("4"),
    source="NASDAQ",
)

# Simulated daily bars around the split
pre_split_bar = Bar(
    instrument_id="INT-000001",
    bar_start=datetime(2020, 8, 28, 13, 30, tzinfo=timezone.utc),
    bar_end=datetime(2020, 8, 28, 20, 0, tzinfo=timezone.utc),
    open=Decimal("500.00"), high=Decimal("505.00"),
    low=Decimal("498.00"), close=Decimal("503.00"),
    volume=Decimal("50000"), vwap=Decimal("501.50"),
    trade_count=12000,
)
post_split_bar = Bar(
    instrument_id="INT-000001",
    bar_start=datetime(2020, 8, 31, 13, 30, tzinfo=timezone.utc),
    bar_end=datetime(2020, 8, 31, 20, 0, tzinfo=timezone.utc),
    open=Decimal("127.58"), high=Decimal("131.00"),
    low=Decimal("126.00"), close=Decimal("129.04"),
    volume=Decimal("225000"), vwap=Decimal("128.50"),
    trade_count=45000,
)

print("=== Unadjusted bars (looks like a crash!) ===")
print(f"  Aug 28 close: ${pre_split_bar.close}  volume: {pre_split_bar.volume:,}")
print(f"  Aug 31 close: ${post_split_bar.close}  volume: {post_split_bar.volume:,}")
print(f"  Apparent return: {((post_split_bar.close - pre_split_bar.close) / pre_split_bar.close * 100):.1f}%")
print()

adjusted_bars = adjust_bars_for_split([pre_split_bar, post_split_bar], split)
print("=== Adjusted bars (continuous price series) ===")
print(f"  Aug 28 close: ${adjusted_bars[0].close}  volume: {adjusted_bars[0].volume:,}")
print(f"  Aug 31 close: ${adjusted_bars[1].close}  volume: {adjusted_bars[1].volume:,}")
print(f"  Actual return: {((adjusted_bars[1].close - adjusted_bars[0].close) / adjusted_bars[0].close * 100):.1f}%")

# Try next:
# 1. Implement adjust_bars_for_dividend that adjusts for a $0.50 cash dividend.
# 2. What happens if you compute a 5-day moving average across the unadjusted split?
# 3. Why are returns (percent changes) naturally split-invariant?

## 2. Symbology and Canonical Identifiers

Every data vendor uses different symbols for the same instrument. This is not a minor annoyance — it is a fundamental data engineering problem that, if solved badly, causes misrouted orders, broken joins, and corrupted analytics.

**The problem in concrete terms:**

| Vendor | Apple symbol | Format |
|--------|-------------|--------|
| Bloomberg | `AAPL US Equity` | ticker + country + asset class |
| Reuters/Refinitiv | `AAPL.O` | ticker + exchange suffix |
| NYSE (exchange) | `AAPL` | bare ticker |
| ISIN | `US0378331005` | country + NSIN + check digit |
| FIGI | `BBG000B9XRY4` | opaque 12-char identifier |

And it gets worse. Tickers are reused. The ticker `TWTR` was Twitter, then it was delisted, and the letters could be reassigned to a completely different company. The ticker `META` belonged to a small company called Roundhill Ball Metaverse ETF before Facebook claimed it.

**The solution: internal canonical ID + symbol mapping table.** Your platform assigns each instrument a stable internal identifier (e.g., `INT-000001`). A mapping table translates between your internal ID and every vendor's symbol. When data arrives from any vendor, the first step is to resolve the vendor symbol to your canonical ID. This is the "symbology layer."

**Industry standards worth knowing:**
- **ISIN**: the closest thing to a global standard. Required for regulatory reporting in most jurisdictions. 12 characters: 2-letter country code + 9-char NSIN + check digit.
- **FIGI (OpenFIGI)**: Bloomberg's open identifier standard. Free to use, covers global instruments. 12 characters, opaque.
- **CUSIP**: North American standard, maintained by S&P. 9 characters. Embedded in ISINs for US/Canadian securities.
- **SEDOL**: London Stock Exchange standard. 7 characters. Embedded in ISINs for UK/Irish securities.

In FX, the convention is simpler: ISO 4217 currency pair codes (EURUSD, USDJPY). But even here, quoting conventions differ — some platforms quote EUR/USD, others USD/EUR, and the spread conventions vary.

In [ ]:
@dataclass
class SymbolMapping:
    """Maps a vendor-specific symbol to the internal canonical ID."""

    internal_id: str
    vendor: str
    vendor_symbol: str
    effective_start: datetime
    effective_end: Optional[datetime] = None  # None = currently active


class SymbolMapper:
    """Resolves vendor symbols to canonical internal IDs.

    This is the core of the symbology layer. Every piece of incoming data
    passes through this mapper before it touches any downstream system.
    """

    def __init__(self, mappings: list[SymbolMapping]) -> None:
        # Index: (vendor, vendor_symbol) -> list of mappings (for SCD2 time lookup)
        self._index: dict[tuple[str, str], list[SymbolMapping]] = {}
        for m in mappings:
            key = (m.vendor.upper(), m.vendor_symbol.upper())
            self._index.setdefault(key, []).append(m)

    def resolve(self, vendor: str, vendor_symbol: str, as_of: Optional[datetime] = None) -> Optional[str]:
        """Resolve a vendor symbol to the canonical internal ID.

        Args:
            vendor: the data vendor name
            vendor_symbol: the symbol as the vendor reports it
            as_of: point-in-time lookup (default: now)

        Returns:
            The internal_id, or None if no mapping exists.
        """
        key = (vendor.upper(), vendor_symbol.upper())
        candidates = self._index.get(key, [])

        if as_of is None:
            as_of = datetime.now(timezone.utc)

        for m in candidates:
            if m.effective_start <= as_of and (m.effective_end is None or as_of < m.effective_end):
                return m.internal_id
        return None

    def reverse_lookup(self, internal_id: str, vendor: str) -> Optional[str]:
        """Given an internal ID and vendor, find the current vendor symbol."""
        now = datetime.now(timezone.utc)
        for key, mappings in self._index.items():
            if key[0] == vendor.upper():
                for m in mappings:
                    if m.internal_id == internal_id and m.effective_start <= now and (m.effective_end is None or now < m.effective_end):
                        return m.vendor_symbol
        return None


# Build a symbol mapping table
mappings = [
    # Apple across vendors
    SymbolMapping("INT-000001", "BLOOMBERG", "AAPL US Equity", datetime(2020, 1, 1, tzinfo=timezone.utc)),
    SymbolMapping("INT-000001", "REUTERS", "AAPL.O", datetime(2020, 1, 1, tzinfo=timezone.utc)),
    SymbolMapping("INT-000001", "EXCHANGE", "AAPL", datetime(2020, 1, 1, tzinfo=timezone.utc)),

    # Facebook -> Meta rename (SCD2 in symbology!)
    SymbolMapping("INT-000042", "BLOOMBERG", "FB US Equity", datetime(2012, 5, 18, tzinfo=timezone.utc),
                  effective_end=datetime(2022, 6, 9, tzinfo=timezone.utc)),
    SymbolMapping("INT-000042", "BLOOMBERG", "META US Equity", datetime(2022, 6, 9, tzinfo=timezone.utc)),
    SymbolMapping("INT-000042", "EXCHANGE", "FB", datetime(2012, 5, 18, tzinfo=timezone.utc),
                  effective_end=datetime(2022, 6, 9, tzinfo=timezone.utc)),
    SymbolMapping("INT-000042", "EXCHANGE", "META", datetime(2022, 6, 9, tzinfo=timezone.utc)),

    # EURUSD across vendors
    SymbolMapping("INT-FX-0001", "BLOOMBERG", "EURUSD Curncy", datetime(2020, 1, 1, tzinfo=timezone.utc)),
    SymbolMapping("INT-FX-0001", "REUTERS", "EUR=", datetime(2020, 1, 1, tzinfo=timezone.utc)),
    SymbolMapping("INT-FX-0001", "EBS", "EURUSD", datetime(2020, 1, 1, tzinfo=timezone.utc)),
]

mapper = SymbolMapper(mappings)

# Resolve vendor symbols to canonical IDs
print("=== Symbol Resolution ===")
tests = [
    ("BLOOMBERG", "AAPL US Equity"),
    ("REUTERS", "AAPL.O"),
    ("EXCHANGE", "AAPL"),
    ("BLOOMBERG", "EURUSD Curncy"),
    ("REUTERS", "EUR="),
    ("EBS", "EURUSD"),
]
for vendor, sym in tests:
    result = mapper.resolve(vendor, sym)
    print(f"  {vendor:12s} {sym:20s} -> {result}")

print()
print("=== Time-aware resolution (FB vs META) ===")
pre_rename = datetime(2021, 1, 1, tzinfo=timezone.utc)
post_rename = datetime(2023, 1, 1, tzinfo=timezone.utc)
print(f"  Bloomberg 'FB US Equity'   @ 2021: {mapper.resolve('BLOOMBERG', 'FB US Equity', pre_rename)}")
print(f"  Bloomberg 'FB US Equity'   @ 2023: {mapper.resolve('BLOOMBERG', 'FB US Equity', post_rename)}")
print(f"  Bloomberg 'META US Equity' @ 2023: {mapper.resolve('BLOOMBERG', 'META US Equity', post_rename)}")

print()
print("=== Reverse lookup ===")
print(f"  INT-000001 on Bloomberg: {mapper.reverse_lookup('INT-000001', 'BLOOMBERG')}")
print(f"  INT-FX-0001 on Reuters: {mapper.reverse_lookup('INT-FX-0001', 'REUTERS')}")

# Try next:
# 1. Add USDJPY mappings for Bloomberg ("USDJPY Curncy"), Reuters ("JPY="), and EBS.
# 2. What should happen when resolve() returns None? (unmapped symbol = data quality alert)
# 3. Add ISIN as a "vendor" and map INT-000001 to US0378331005.

## 3. Venue Calendars and Session Boundaries

Markets are not open 24/7/365 (with the partial exception of FX and crypto). Knowing when a market is open is not optional — it is load-bearing for:

- **Daily bar computation:** A "daily bar" for NYSE means 9:30-16:00 ET. A "daily bar" for LSE means 8:00-16:30 GMT. If you compute bars using UTC midnight boundaries, you get garbage.
- **Trade filtering:** Pre-market and after-hours trades have different characteristics (wider spreads, lower volume, more noise). Many analytics explicitly exclude them.
- **Anomaly detection:** A trade arriving on Christmas Day for NYSE is either an error or needs special handling.
- **Feature engineering:** "Volume relative to session average" requires knowing when the session is.

**Key venue hours (regular session only):**

| Venue | Local hours | UTC equivalent |
|-------|------------|----------------|
| NYSE/NASDAQ | 9:30-16:00 ET | 13:30-20:00 (EST) or 13:30-20:00 (EDT) |
| LSE | 8:00-16:30 GMT | 8:00-16:30 (winter) or 7:00-15:30 (summer) |
| TSE (Tokyo) | 9:00-11:30, 12:30-15:00 JST | 0:00-2:30, 3:30-6:00 |
| FX Spot | 24h Sun 17:00 ET - Fri 17:00 ET | Continuous except weekends |

**The timezone trap:** DST transitions move the UTC offset. NYSE opens at 13:30 UTC in winter but 13:30 UTC in summer too — wait, no. It opens at 9:30 ET always, which is 14:30 UTC in winter (EST = UTC-5) and 13:30 UTC in summer (EDT = UTC-4). If you hardcode UTC offsets instead of using proper timezone-aware datetime handling, you will have bugs twice a year.

**The rule: store everything in UTC, convert to local time only for display and session-boundary logic using proper timezone libraries.**

In [ ]:
from zoneinfo import ZoneInfo
from datetime import time as dt_time


@dataclass
class SessionBoundary:
    """Defines when a trading session is open for a specific venue."""

    venue: str
    timezone: str  # IANA timezone (e.g., "America/New_York")
    open_time: dt_time  # local time
    close_time: dt_time  # local time
    days: tuple[int, ...] = (0, 1, 2, 3, 4)  # Monday=0 through Friday=4
    session_type: str = "REGULAR"  # REGULAR, PRE_MARKET, AFTER_HOURS


@dataclass
class VenueCalendar:
    """Calendar for a trading venue with session boundaries and holidays."""

    venue: str
    sessions: list[SessionBoundary]
    holidays: set[str]  # ISO date strings: {"2026-12-25", "2026-01-01", ...}

    def is_open(self, utc_time: datetime, session_type: str = "REGULAR") -> bool:
        """Check if the venue is open at a given UTC time."""
        for session in self.sessions:
            if session.session_type != session_type:
                continue

            # Convert UTC to venue local time
            local_tz = ZoneInfo(session.timezone)
            local_time = utc_time.astimezone(local_tz)

            # Check day of week (Monday=0)
            if local_time.weekday() not in session.days:
                continue

            # Check holidays
            date_str = local_time.strftime("%Y-%m-%d")
            if date_str in self.holidays:
                continue

            # Check time bounds
            if session.open_time <= local_time.time() < session.close_time:
                return True

        return False

    def session_bounds_utc(self, date: datetime, session_type: str = "REGULAR") -> Optional[tuple[datetime, datetime]]:
        """Get UTC open/close times for a given date."""
        for session in self.sessions:
            if session.session_type != session_type:
                continue

            local_tz = ZoneInfo(session.timezone)
            local_date = date.astimezone(local_tz).date()

            # Check day of week
            if local_date.weekday() not in session.days:
                continue

            # Check holidays
            if local_date.isoformat() in self.holidays:
                continue

            # Build local open/close, then convert to UTC
            local_open = datetime.combine(local_date, session.open_time, tzinfo=local_tz)
            local_close = datetime.combine(local_date, session.close_time, tzinfo=local_tz)
            return (local_open.astimezone(timezone.utc), local_close.astimezone(timezone.utc))

        return None


# Define venue calendars
nyse_calendar = VenueCalendar(
    venue="NYSE",
    sessions=[
        SessionBoundary("NYSE", "America/New_York", dt_time(9, 30), dt_time(16, 0), session_type="REGULAR"),
        SessionBoundary("NYSE", "America/New_York", dt_time(4, 0), dt_time(9, 30), session_type="PRE_MARKET"),
        SessionBoundary("NYSE", "America/New_York", dt_time(16, 0), dt_time(20, 0), session_type="AFTER_HOURS"),
    ],
    holidays={"2026-01-01", "2026-01-19", "2026-07-03", "2026-12-25"},
)

lse_calendar = VenueCalendar(
    venue="LSE",
    sessions=[
        SessionBoundary("LSE", "Europe/London", dt_time(8, 0), dt_time(16, 30), session_type="REGULAR"),
    ],
    holidays={"2026-01-01", "2026-12-25", "2026-12-28"},
)

fx_calendar = VenueCalendar(
    venue="FX",
    sessions=[
        # FX is 24h Sun evening to Fri evening — simplify as Mon-Fri 00:00-23:59 UTC
        SessionBoundary("FX", "UTC", dt_time(0, 0), dt_time(23, 59, 59), session_type="REGULAR"),
    ],
    holidays=set(),  # FX has no holidays (though liquidity drops)
)

# Test session boundaries
test_times = [
    ("NYSE regular hours", datetime(2026, 4, 6, 14, 0, 0, tzinfo=timezone.utc)),  # Monday 10:00 ET
    ("NYSE before open", datetime(2026, 4, 6, 12, 0, 0, tzinfo=timezone.utc)),   # Monday 8:00 ET
    ("NYSE weekend", datetime(2026, 4, 4, 14, 0, 0, tzinfo=timezone.utc)),       # Saturday
    ("NYSE holiday", datetime(2026, 1, 1, 14, 0, 0, tzinfo=timezone.utc)),       # New Year's Day
    ("LSE regular hours", datetime(2026, 4, 6, 10, 0, 0, tzinfo=timezone.utc)),  # Monday 10:00 GMT
]

print("=== Session Boundary Checks ===")
for label, t in test_times:
    nyse_open = nyse_calendar.is_open(t)
    lse_open = lse_calendar.is_open(t)
    print(f"  {label:25s} ({t.strftime('%a %H:%M UTC')}): NYSE={'OPEN' if nyse_open else 'CLOSED':6s} LSE={'OPEN' if lse_open else 'CLOSED'}")

print()
print("=== UTC Session Bounds for NYSE on 2026-04-06 ===")
bounds = nyse_calendar.session_bounds_utc(datetime(2026, 4, 6, 12, 0, tzinfo=timezone.utc))
if bounds:
    print(f"  Open:  {bounds[0].isoformat()}")
    print(f"  Close: {bounds[1].isoformat()}")

# Try next:
# 1. Add a TSE (Tokyo) calendar with the split lunch session.
# 2. Filter a list of trades to only those within NYSE regular hours.
# 3. What happens during the DST transition in March? Test with a date in both EST and EDT.

## 4. Time Concepts: Event Time vs Processing Time vs As-Of Time

This is where trading data engineering diverges sharply from generic data engineering. In a typical web application, "when did this happen?" has one answer. In trading data, there are three distinct answers, and confusing them is the root cause of most correctness bugs.

### The three timestamps

**Event time** — when the thing actually happened in the real world. A trade executed on the exchange at 10:00:00.123 UTC has that as its event time. This is the "truth" timestamp, but you do not control it and you may not receive it promptly.

**Processing time (ingest time)** — when your system received or processed the event. If the trade arrived via Kafka at 10:00:00.456 UTC, that is the processing time. In batch pipelines, this could be hours or even a day later. Processing time is the only timestamp you fully control.

**As-of time (knowledge time)** — the point in time at which you are asking a question about the data. "What was the VWAP for EURUSD at 10:00?" is an as-of query. The answer depends on what data had arrived by 10:00. If a late trade from 09:58 arrives at 10:05, the "as-of 10:00" VWAP does not include it, but the "as-of 10:06" VWAP does.

### Why they diverge

- **Network latency:** Exchange to your Kafka cluster is not instantaneous (typically 1-100ms for co-located systems, seconds for cloud).
- **Batch delays:** End-of-day files from some vendors arrive hours after market close.
- **Corrections:** An exchange discovers it reported a wrong price and sends a correction 2 hours later.
- **Late arrivals:** A trade from venue A arrives 30 seconds late because the venue's gateway had a hiccup.
- **Out-of-order delivery:** Kafka guarantees ordering within a partition, but if trades from different venues are on different partitions, they can arrive out of event-time order.

### The consequence for windowed aggregation

If you compute a 1-minute VWAP window [10:00, 10:01) and a trade with event_time=10:00:30 arrives at 10:02:00, what do you do?

- **Option 1:** Ignore it (the window is closed). Simple but loses data.
- **Option 2:** Recompute the window. Correct but expensive and creates downstream update storms.
- **Option 3:** Grace period — keep the window open for an extra N seconds to absorb late arrivals. This is what Kafka Streams and Flink do. It is a tradeoff between latency and completeness.

There is no universally right answer. The choice depends on your SLA for completeness vs latency.

In [ ]:
@dataclass(frozen=True)
class TimestampedTrade:
    """A trade with all three timestamp dimensions explicitly modeled."""

    trade_id: str
    instrument_id: str
    price: Decimal
    size: Decimal
    side: Side
    event_time: datetime  # when the trade happened on the exchange
    ingest_time: datetime  # when our Kafka consumer received it
    as_of_time: datetime  # when the record became "known" in our system

    @property
    def latency_ms(self) -> float:
        """Network/processing latency: ingest - event."""
        return (self.ingest_time - self.event_time).total_seconds() * 1000

    @property
    def staleness_ms(self) -> float:
        """Total staleness: as_of - event."""
        return (self.as_of_time - self.event_time).total_seconds() * 1000


# Scenario: Three trades illustrating timestamp divergence
base = datetime(2026, 4, 4, 10, 0, 0, tzinfo=timezone.utc)

# Normal trade: small latency, everything aligns
normal = TimestampedTrade(
    trade_id="T1", instrument_id="INT-FX-0001",
    price=Decimal("1.08450"), size=Decimal("2000000"), side=Side.BUY,
    event_time=base,
    ingest_time=base + timedelta(milliseconds=5),
    as_of_time=base + timedelta(milliseconds=10),
)

# Late arrival: exchange gateway hiccup, trade arrives 30 seconds late
late = TimestampedTrade(
    trade_id="T2", instrument_id="INT-FX-0001",
    price=Decimal("1.08455"), size=Decimal("3000000"), side=Side.SELL,
    event_time=base + timedelta(seconds=15),
    ingest_time=base + timedelta(seconds=45),  # 30 seconds late!
    as_of_time=base + timedelta(seconds=46),
)

# Correction: exchange sends corrected price 2 hours later
correction = TimestampedTrade(
    trade_id="T3-CORRECTED", instrument_id="INT-FX-0001",
    price=Decimal("1.08448"),  # corrected from 1.08450
    size=Decimal("2000000"), side=Side.BUY,
    event_time=base,  # same event time as original
    ingest_time=base + timedelta(hours=2),  # arrived 2 hours later
    as_of_time=base + timedelta(hours=2, milliseconds=50),
)

print("=== Three Timestamp Dimensions ===")
print()
for label, t in [("Normal", normal), ("Late arrival", late), ("Correction", correction)]:
    print(f"{label} ({t.trade_id}):")
    print(f"  event_time:  {t.event_time.strftime('%H:%M:%S.%f')[:-3]}")
    print(f"  ingest_time: {t.ingest_time.strftime('%H:%M:%S.%f')[:-3]}")
    print(f"  as_of_time:  {t.as_of_time.strftime('%H:%M:%S.%f')[:-3]}")
    print(f"  latency:     {t.latency_ms:,.0f}ms")
    print(f"  staleness:   {t.staleness_ms:,.0f}ms")
    print()

# The critical question: which window does each trade belong to?
window_start = base
window_end = base + timedelta(minutes=1)
grace_period = timedelta(seconds=30)

print("=== Window Assignment [10:00:00, 10:01:00) with 30s grace ===")
for label, t in [("Normal", normal), ("Late arrival", late), ("Correction", correction)]:
    in_window = window_start <= t.event_time < window_end
    arrived_in_time = t.ingest_time < window_end + grace_period
    included = in_window and arrived_in_time
    print(f"  {label:15s}: event_in_window={in_window}, arrived_in_grace={arrived_in_time}, INCLUDED={included}")

# Try next:
# 1. What happens to the correction if the grace period is only 5 seconds?
# 2. Build a function that partitions trades into "on-time", "late but within grace", and "too late".
# 3. If you query "VWAP as-of 10:00:30", which of these three trades should be included?

## 5. Point-in-Time Correctness

**This is the single most important concept in this entire notebook.** If you take away one thing, let it be this.

### What it means

Point-in-time correctness means: when you look back at what you "knew" at time T, you should see exactly the information that was available at time T — nothing more, nothing less. No future data leaking backward. No corrections retroactively applied. The exact state of knowledge at that moment.

### Why it matters

Imagine you are backtesting a trading strategy. At 10:00 on Monday, the strategy decides to buy EURUSD based on the last 20 minutes of trade data. If your backtesting system uses data that includes a correction that arrived at 14:00 on Monday, the backtest is using information the strategy could not have had at 10:00. The backtest results are wrong — they are better than what the strategy would have actually achieved, because you are giving it perfect information that was not available in real time.

This is called **lookahead bias** or **future information leakage**, and it is the number one killer of backtesting credibility. Funds have allocated millions of dollars to strategies that looked great in backtesting but failed in production, because the backtests were not point-in-time correct.

### Implementation: as-of joins

The standard implementation technique is the **as-of join** (also called a temporal join or point-in-time join). When joining features to labels:

- For each prediction point at time T, join only features where `as_of_time <= T`
- Take the most recent feature value that was available before T
- Never join a feature that has `as_of_time > T`

### Bi-temporal tables

The most rigorous implementation uses **bi-temporal tables** with two time dimensions:

- **valid_time**: when the fact was true in the real world (event time)
- **transaction_time**: when the system recorded it (as-of time)

To query "what did we know at time T about events up to time E?", you filter on both: `valid_time <= E AND transaction_time <= T`. This handles corrections correctly — the original record has transaction_time=T1, the correction has transaction_time=T2>T1, so an as-of-T1 query sees the original and an as-of-T2 query sees the correction.

In [ ]:
@dataclass(frozen=True)
class BiTemporalRecord:
    """A record with both valid_time and transaction_time dimensions."""

    record_id: str
    instrument_id: str
    field_name: str
    value: Decimal
    valid_time: datetime  # when the fact was true in the real world
    transaction_time: datetime  # when the system recorded this version
    is_correction: bool = False


class BiTemporalStore:
    """Simple bi-temporal store demonstrating point-in-time queries."""

    def __init__(self) -> None:
        self._records: list[BiTemporalRecord] = []

    def insert(self, record: BiTemporalRecord) -> None:
        self._records.append(record)

    def query_as_of(
        self,
        instrument_id: str,
        field_name: str,
        valid_up_to: datetime,
        knowledge_as_of: datetime,
    ) -> Optional[BiTemporalRecord]:
        """Point-in-time query: what did we know at knowledge_as_of about events up to valid_up_to?

        Returns the most recent record that satisfies BOTH time constraints.
        """
        candidates = [
            r for r in self._records
            if r.instrument_id == instrument_id
            and r.field_name == field_name
            and r.valid_time <= valid_up_to
            and r.transaction_time <= knowledge_as_of
        ]

        if not candidates:
            return None

        # Return the most recent by transaction_time (latest knowledge)
        return max(candidates, key=lambda r: (r.valid_time, r.transaction_time))


# Demonstrate point-in-time correctness with a correction scenario
store = BiTemporalStore()
base = datetime(2026, 4, 4, 10, 0, 0, tzinfo=timezone.utc)

# Original VWAP published at 10:01 for the 10:00 window
store.insert(BiTemporalRecord(
    record_id="VWAP-001",
    instrument_id="INT-FX-0001",
    field_name="vwap_1min",
    value=Decimal("1.08450"),
    valid_time=base,
    transaction_time=base + timedelta(minutes=1),
))

# Correction published at 12:00 — a late trade changed the VWAP
store.insert(BiTemporalRecord(
    record_id="VWAP-001-CORRECTED",
    instrument_id="INT-FX-0001",
    field_name="vwap_1min",
    value=Decimal("1.08453"),  # slightly different
    valid_time=base,  # same event window
    transaction_time=base + timedelta(hours=2),  # correction arrived later
    is_correction=True,
))

print("=== Point-in-Time Queries ===")
print()

# Query 1: What did we know at 10:05? (before correction)
result_early = store.query_as_of(
    instrument_id="INT-FX-0001",
    field_name="vwap_1min",
    valid_up_to=base,
    knowledge_as_of=base + timedelta(minutes=5),
)
print(f"As-of 10:05 (before correction):")
print(f"  VWAP = {result_early.value}  (original)")
print()

# Query 2: What did we know at 13:00? (after correction)
result_late = store.query_as_of(
    instrument_id="INT-FX-0001",
    field_name="vwap_1min",
    valid_up_to=base,
    knowledge_as_of=base + timedelta(hours=3),
)
print(f"As-of 13:00 (after correction):")
print(f"  VWAP = {result_late.value}  (corrected)")
print()

print("Same event, same window, different answers depending on WHEN you ask.")
print(f"Difference: {result_late.value - result_early.value} (3 pips in FX terms)")
print()

# Demonstrate the as-of join for backtesting
print("=== As-Of Join for Backtesting ===")
print()

# Feature values that arrived at different times
features = [
    {"feature": "spread_20s_avg", "value": Decimal("0.00005"), "as_of": base + timedelta(seconds=20)},
    {"feature": "spread_20s_avg", "value": Decimal("0.00006"), "as_of": base + timedelta(seconds=40)},
    {"feature": "spread_20s_avg", "value": Decimal("0.00004"), "as_of": base + timedelta(seconds=60)},
]

# Prediction point: 10:00:45 — what features were available?
prediction_time = base + timedelta(seconds=45)
available = [f for f in features if f["as_of"] <= prediction_time]
latest = max(available, key=lambda f: f["as_of"]) if available else None

print(f"Prediction at {prediction_time.strftime('%H:%M:%S')}:")
print(f"  Available features: {len(available)} of {len(features)}")
print(f"  Latest feature value: {latest['value']} (as_of {latest['as_of'].strftime('%H:%M:%S')})")
print(f"  Feature at :60 ({features[2]['value']}) is EXCLUDED — it is in the future!")

# Try next:
# 1. Add a second correction at 14:00 and verify the as-of query returns the right version.
# 2. Implement a function that builds a complete feature vector for a list of prediction times.
# 3. What happens if you accidentally use event_time instead of as_of_time for the join?

## 6. Leakage Prevention

Data leakage is when information from outside the training set — typically from the future or from the test set — sneaks into the model's training process. In trading, leakage is insidious because it makes bad strategies look good. A model trained with leakage will show excellent backtest performance and then lose money in production.

### Common leakage sources in trading data

**1. Using the close price to predict the close price.** This sounds obvious but happens more often than you think — for example, computing a "daily return" feature using today's close and then predicting whether today's close is higher than yesterday's. The feature already contains the answer.

**2. Features computed from the full dataset.** If you normalize features using the mean and standard deviation of the entire dataset (including the test period), the test-period statistics leak into the training set. The model "knows" the future distribution.

**3. Using corrected/adjusted data that was not available at prediction time.** If a trade was corrected at 14:00 but your model was making predictions at 10:00, using the corrected value in training teaches the model to use information it cannot have in production.

**4. Random train/test split on time-series data.** This is the most common and most dangerous mistake. If you randomly split time-series data, future observations end up in the training set and past observations end up in the test set. The model learns temporal patterns from the future.

### Why random split is wrong for time-series

In cross-sectional data (e.g., predicting house prices from features), observations are approximately independent. Random splitting is fine because knowing one house's price does not help you predict another's.

In time-series data, observations are **autocorrelated** — today's price depends on yesterday's price. If you train on data from Tuesday and test on Monday, you are literally training on the future. Even if you are not that extreme, temporal patterns (trends, seasonality, regime changes) mean that nearby-in-time observations share information. Random splitting breaks the temporal structure.

### The correct approach: temporal train/validate/test split

```
|---- TRAIN ----|-- VALIDATE --|---- TEST ----|
t=0            T             T+V           T+V+E

- Train on [0, T]
- Validate on (T, T+V]
- Test on (T+V, T+V+E]
```

Optionally, use a **gap** between train and validate/test to avoid short-term autocorrelation leaking across the boundary. For daily data, a gap of 1-5 days is common.

In [ ]:
import random
from collections import defaultdict as ddict


def generate_mock_price_series(n_days: int, start_price: Decimal, seed: int = 42) -> list[dict]:
    """Generate a mock daily price series with trend + noise.

    Each row: {date, price, return_pct, day_index}
    The series has a slight upward trend embedded, which a model should be able to detect.
    """
    random.seed(seed)
    rows = []
    price = float(start_price)
    base_date = datetime(2025, 1, 1, tzinfo=timezone.utc)

    for i in range(n_days):
        daily_return = 0.0003 + random.gauss(0, 0.01)  # slight positive drift + noise
        price *= (1 + daily_return)
        rows.append({
            "date": base_date + timedelta(days=i),
            "price": Decimal(str(round(price, 5))),
            "return_pct": round(daily_return * 100, 4),
            "day_index": i,
        })

    return rows


# Generate 500 trading days of mock data
series = generate_mock_price_series(500, Decimal("1.08000"))
print(f"Generated {len(series)} daily observations")
print(f"  First: {series[0]['date'].date()} price={series[0]['price']}")
print(f"  Last:  {series[-1]['date'].date()} price={series[-1]['price']}")
print()


# === WRONG: Random split ===
random.seed(99)
indices = list(range(len(series)))
random.shuffle(indices)

random_train_idx = sorted(indices[:350])
random_test_idx = sorted(indices[350:])

# Check: are there future dates in the training set?
random_train_dates = [series[i]["date"] for i in random_train_idx]
random_test_dates = [series[i]["date"] for i in random_test_idx]

future_in_train = sum(1 for td in random_train_dates if td > min(random_test_dates))
past_in_test = sum(1 for td in random_test_dates if td < max(random_train_dates))

print("=== WRONG: Random Split ===")
print(f"  Train: {len(random_train_idx)} samples, dates from {min(random_train_dates).date()} to {max(random_train_dates).date()}")
print(f"  Test:  {len(random_test_idx)} samples, dates from {min(random_test_dates).date()} to {max(random_test_dates).date()}")
print(f"  LEAKAGE: {future_in_train} training samples are AFTER the earliest test sample!")
print(f"  LEAKAGE: {past_in_test} test samples are BEFORE the latest training sample!")
print(f"  Train and test date ranges OVERLAP — the model sees the future during training.")
print()


# === CORRECT: Temporal split ===
train_end = 350
gap = 5  # 5-day gap to avoid autocorrelation leakage
val_start = train_end + gap
val_end = val_start + 70
test_start = val_end + gap

temporal_train = series[:train_end]
temporal_val = series[val_start:val_end]
temporal_test = series[test_start:]

print("=== CORRECT: Temporal Split with Gap ===")
print(f"  Train:    {len(temporal_train)} samples, {temporal_train[0]['date'].date()} to {temporal_train[-1]['date'].date()}")
print(f"  (gap:     {gap} days)")
print(f"  Validate: {len(temporal_val)} samples, {temporal_val[0]['date'].date()} to {temporal_val[-1]['date'].date()}")
print(f"  (gap:     {gap} days)")
print(f"  Test:     {len(temporal_test)} samples, {temporal_test[0]['date'].date()} to {temporal_test[-1]['date'].date()}")
print(f"  NO OVERLAP: max train date < min validate date < min test date")
print()


# === Demonstrate normalization leakage ===
all_returns = [row["return_pct"] for row in series]
train_returns = [row["return_pct"] for row in temporal_train]

global_mean = sum(all_returns) / len(all_returns)
global_std = (sum((r - global_mean) ** 2 for r in all_returns) / len(all_returns)) ** 0.5

train_mean = sum(train_returns) / len(train_returns)
train_std = (sum((r - train_mean) ** 2 for r in train_returns) / len(train_returns)) ** 0.5

print("=== Normalization Leakage ===")
print(f"  Global mean (WRONG): {global_mean:.4f}  std: {global_std:.4f}")
print(f"  Train mean (RIGHT):  {train_mean:.4f}  std: {train_std:.4f}")
print(f"  Difference in mean:  {abs(global_mean - train_mean):.4f}")
print(f"  Using global stats leaks test-period distribution into training features.")

# Try next:
# 1. Compute accuracy of a naive "predict up if return > mean" model under both splits.
# 2. Add walk-forward validation: retrain every 50 days, predict the next 10.
# 3. What if the gap is 0? Can autocorrelation still leak?

## 7. Feature Engineering for Trading Data

Feature engineering in trading is constrained by one absolute rule: **every feature must use only data that was available at the time of prediction.** This means every feature has an explicit lookback window — a window that extends backward from the current point, never forward.

### Lookback windows

A "20-day moving average" means: at each point t, average the last 20 values from t-19 to t (inclusive). Not the surrounding 20. Not the next 20. The **trailing** 20.

This seems obvious, but many standard data-science tools (pandas `rolling`, numpy `convolve`) can be configured to center the window or include future values by default. You must be explicit.

### Common trading features

| Feature | Formula | What it captures |
|---------|---------|-----------------|
| **Return** | (p_t - p_{t-1}) / p_{t-1} | Price change as a percentage |
| **Log return** | ln(p_t / p_{t-1}) | Additive over time, better for modeling |
| **Volatility** | std(returns over trailing N) | Risk/uncertainty |
| **VWAP deviation** | (price - VWAP) / VWAP | How far price is from fair value |
| **Spread** | ask - bid | Liquidity cost |
| **Volume ratio** | volume_t / avg(volume over trailing N) | Relative activity level |
| **Momentum (RSI-like)** | avg(up returns) / avg(abs(returns)) over N | Trend strength |
| **Bid-ask imbalance** | (bid_size - ask_size) / (bid_size + ask_size) | Supply/demand pressure |

### Feature hygiene rules

1. **No future data.** Every window is strictly trailing.
2. **Explicit NaN handling.** At the start of the series, you do not have N periods of history. These rows should be marked as NaN, not filled with zeros or forward-filled from the future.
3. **Compute on the training set only.** Statistics used for normalization (mean, std) come from the training set and are applied unchanged to validation and test.
4. **Document the lookback.** Every feature must declare its lookback window length. A downstream consumer needs to know: "to compute this feature, I need at least 20 days of history."

In [ ]:
import math


def compute_features(
    series: list[dict],
    lookback: int = 20,
) -> list[dict]:
    """Compute trading features with explicit lookback windows.

    Every feature uses ONLY data from [i - lookback + 1, i] (trailing window).
    Rows without enough history get None — never backfill or use future data.

    Features computed:
    1. return_1d: single-period return
    2. volatility_{lookback}d: rolling standard deviation of returns
    3. momentum_{lookback}d: average of positive returns / average of absolute returns
    4. volume_ratio_{lookback}d: current volume / trailing average volume
    5. price_vs_ma_{lookback}d: (price - trailing MA) / trailing MA
    """
    results = []

    for i, row in enumerate(series):
        features: dict = {
            "date": row["date"],
            "price": row["price"],
        }

        # Feature 1: 1-day return (needs 1 lookback period)
        if i >= 1:
            prev_price = float(series[i - 1]["price"])
            curr_price = float(row["price"])
            features["return_1d"] = round((curr_price - prev_price) / prev_price, 6)
        else:
            features["return_1d"] = None  # not enough history

        # Features 2-5: need `lookback` periods of history
        if i >= lookback:
            # Get trailing window of returns (already computed in series as return_pct)
            window_returns = [series[j]["return_pct"] / 100 for j in range(i - lookback + 1, i + 1)]
            window_prices = [float(series[j]["price"]) for j in range(i - lookback + 1, i + 1)]

            # Feature 2: Volatility (rolling std of returns)
            mean_ret = sum(window_returns) / len(window_returns)
            var = sum((r - mean_ret) ** 2 for r in window_returns) / len(window_returns)
            features[f"volatility_{lookback}d"] = round(math.sqrt(var), 6)

            # Feature 3: Momentum (proportion of returns that are positive, weighted by magnitude)
            up_returns = [r for r in window_returns if r > 0]
            abs_returns = [abs(r) for r in window_returns if r != 0]
            if abs_returns:
                features[f"momentum_{lookback}d"] = round(
                    sum(up_returns) / sum(abs_returns), 4
                )
            else:
                features[f"momentum_{lookback}d"] = 0.5  # neutral

            # Feature 4: Volume ratio (simulated — use price change magnitude as proxy)
            current_activity = abs(window_returns[-1]) if window_returns[-1] != 0 else 0.001
            avg_activity = sum(abs(r) for r in window_returns) / len(window_returns)
            features[f"volume_ratio_{lookback}d"] = round(current_activity / avg_activity, 4) if avg_activity > 0 else 1.0

            # Feature 5: Price vs moving average
            ma = sum(window_prices) / len(window_prices)
            features[f"price_vs_ma_{lookback}d"] = round((float(row["price"]) - ma) / ma, 6)
        else:
            # Not enough history — leave as None (DO NOT fill with 0 or forward-fill)
            features[f"volatility_{lookback}d"] = None
            features[f"momentum_{lookback}d"] = None
            features[f"volume_ratio_{lookback}d"] = None
            features[f"price_vs_ma_{lookback}d"] = None

        results.append(features)

    return results


# Compute features on our mock series
lookback = 20
feature_rows = compute_features(series, lookback=lookback)

print(f"=== Feature Engineering (lookback={lookback}) ===")
print()

# Show first few rows (should have None for insufficient history)
print("First 3 rows (insufficient history — features are None):")
for row in feature_rows[:3]:
    none_count = sum(1 for v in row.values() if v is None)
    print(f"  {row['date'].date()}: price={row['price']}, null_features={none_count}")

print()

# Show row at exactly the lookback boundary
boundary = feature_rows[lookback]
print(f"Row at day {lookback} (first row with all features):")
for k, v in boundary.items():
    if k not in ("date", "price"):
        print(f"  {k}: {v}")

print()

# Show a sample from the middle
sample = feature_rows[250]
print(f"Sample row (day 250, {sample['date'].date()}):")
for k, v in sample.items():
    if k not in ("date", "price"):
        print(f"  {k}: {v}")

print()

# Verify no future data was used
print("=== Leakage Check ===")
print(f"  Rows with all features: {sum(1 for r in feature_rows if r[f'volatility_{lookback}d'] is not None)}")
print(f"  Rows with None features: {sum(1 for r in feature_rows if r[f'volatility_{lookback}d'] is None)}")
print(f"  Expected None rows: {lookback} (need {lookback} periods of history)")
print(f"  Match: {sum(1 for r in feature_rows if r[f'volatility_{lookback}d'] is None) == lookback}")

# Try next:
# 1. Add a "spread" feature using the Quote model from section 1.3.
# 2. What happens if you accidentally use a centered window instead of trailing?
# 3. Implement walk-forward feature computation: recompute features as new data arrives.

## 8. Dataset Versioning and Snapshots

Once you have computed features, you need to freeze them into a **dataset snapshot** — an immutable, versioned view of the data at a specific point in time. Without snapshots, you cannot reproduce experiments.

### Why versioning matters

Consider this scenario: you train a model on Monday, it performs well. On Wednesday, your data pipeline fixes a bug in the VWAP calculation and backfills corrected data. You retrain the same model with the same code — but now the results are different because the underlying data changed.

Without dataset versioning, you cannot:
- Reproduce a past experiment exactly
- Compare two models trained on the same data vs different data
- Audit what data a production model was trained on
- Debug why a model's behavior changed

### What a snapshot captures

A DatasetSnapshot records:
- **snapshot_id**: unique identifier (often a content hash)
- **as_of_time**: the knowledge cutoff — what data was available when the snapshot was created
- **source_versions**: which upstream data sources (and their versions) contributed
- **feature_view_version**: which feature computation logic was used
- **row_count and schema_version**: for quick integrity checks
- **content_hash**: SHA-256 of the actual data, for content-addressable deduplication

### Implementation approaches

1. **Immutable files with metadata sidecar:** Write data as Parquet/CSV with a JSON metadata file. Never overwrite — create new files with new snapshot IDs.
2. **Content-addressable storage:** Hash the data content. If two snapshots have the same hash, they are identical (deduplication for free).
3. **Feature store snapshots:** Tools like Feast, Tecton, and Hopsworks provide built-in snapshot management.

In [ ]:
@dataclass(frozen=True)
class DatasetSnapshot:
    """An immutable, versioned view of a dataset at a specific point in time."""

    snapshot_id: str
    name: str
    as_of_time: datetime  # knowledge cutoff: only data available before this time
    created_at: datetime
    source_versions: dict[str, str]  # {"trades": "v2.1", "quotes": "v1.3"}
    feature_view_name: str
    feature_view_version: str
    row_count: int
    column_names: tuple[str, ...]
    schema_version: str
    content_hash: str  # SHA-256 of the data content
    train_start: Optional[datetime] = None
    train_end: Optional[datetime] = None
    description: str = ""


def create_snapshot(
    name: str,
    feature_rows: list[dict],
    feature_view_name: str,
    feature_view_version: str,
    as_of_time: datetime,
    source_versions: dict[str, str],
    train_start: Optional[datetime] = None,
    train_end: Optional[datetime] = None,
) -> DatasetSnapshot:
    """Create a dataset snapshot from computed feature rows.

    The content hash ensures content-addressable deduplication:
    if two snapshots hash to the same value, they contain identical data.
    """
    # Compute content hash from the actual data
    content_str = json.dumps(
        [{k: str(v) for k, v in row.items()} for row in feature_rows],
        sort_keys=True,
    )
    content_hash = hashlib.sha256(content_str.encode()).hexdigest()

    # Extract column names from the first row
    columns = tuple(feature_rows[0].keys()) if feature_rows else ()

    snapshot_id = f"snap-{content_hash[:12]}"

    return DatasetSnapshot(
        snapshot_id=snapshot_id,
        name=name,
        as_of_time=as_of_time,
        created_at=datetime.now(timezone.utc),
        source_versions=source_versions,
        feature_view_name=feature_view_name,
        feature_view_version=feature_view_version,
        row_count=len(feature_rows),
        column_names=columns,
        schema_version="1.0",
        content_hash=content_hash,
        train_start=train_start,
        train_end=train_end,
    )


# Create a snapshot of our training features
valid_features = [r for r in feature_rows if r[f"volatility_{lookback}d"] is not None]
train_features = valid_features[:330]  # use first 330 valid rows as training data

snapshot = create_snapshot(
    name="eurusd_momentum_features_v1",
    feature_rows=train_features,
    feature_view_name="momentum_features",
    feature_view_version="1.0.0",
    as_of_time=datetime(2026, 4, 4, 12, 0, 0, tzinfo=timezone.utc),
    source_versions={"trades": "mock-v1.0", "quotes": "mock-v1.0"},
    train_start=train_features[0]["date"],
    train_end=train_features[-1]["date"],
)

print("=== Dataset Snapshot ===")
print(f"  snapshot_id:          {snapshot.snapshot_id}")
print(f"  name:                 {snapshot.name}")
print(f"  as_of_time:           {snapshot.as_of_time.isoformat()}")
print(f"  feature_view:         {snapshot.feature_view_name} v{snapshot.feature_view_version}")
print(f"  sources:              {snapshot.source_versions}")
print(f"  rows:                 {snapshot.row_count}")
print(f"  columns:              {snapshot.column_names}")
print(f"  content_hash:         {snapshot.content_hash[:24]}...")
print(f"  train period:         {snapshot.train_start.date()} to {snapshot.train_end.date()}")
print()

# Demonstrate content-addressable deduplication
snapshot_dup = create_snapshot(
    name="eurusd_momentum_features_v1_duplicate",
    feature_rows=train_features,  # same data
    feature_view_name="momentum_features",
    feature_view_version="1.0.0",
    as_of_time=datetime(2026, 4, 4, 12, 0, 0, tzinfo=timezone.utc),
    source_versions={"trades": "mock-v1.0", "quotes": "mock-v1.0"},
)

print("=== Content-Addressable Deduplication ===")
print(f"  Original hash:  {snapshot.content_hash[:24]}...")
print(f"  Duplicate hash: {snapshot_dup.content_hash[:24]}...")
print(f"  Same content?   {snapshot.content_hash == snapshot_dup.content_hash}")

# Try next:
# 1. Modify one row and verify the content hash changes.
# 2. Implement a SnapshotRegistry that stores snapshots and looks them up by ID.
# 3. Serialize the snapshot metadata to JSON for persistent storage.

## 9. FeatureView: Defining Reproducible Feature Sets

A **FeatureView** is a declarative specification of how features are computed from raw data. It answers: "given this raw data, apply these transformations with these parameters to produce these features." The FeatureView is to features what a SQL view is to tables — a named, versioned, reproducible transformation.

### Why FeatureViews matter

If someone asks "how was the `momentum_20d` feature in model v3.2 computed?", the answer should not be "look at the Jupyter notebook from March" or "ask Alex, he wrote the code." The answer should be a versioned FeatureView definition that precisely specifies:

- Which entity (instrument) the features are computed for
- Which input data sources are required
- What lookback window is used
- What the transformation logic is
- What the point-in-time rule is (how late data is handled)

### FeatureView in the ecosystem

This concept comes from the **feature store** world (Feast, Tecton, Hopsworks, Databricks Feature Store). Even if you are not using a feature store, the FeatureView pattern is worth adopting because it forces you to make feature definitions explicit, versioned, and reproducible rather than buried in ad-hoc notebook code.

In [ ]:
class PointInTimeRule(Enum):
    """How to handle late-arriving data in feature computation."""
    STRICT = "STRICT"  # only use data with as_of_time <= prediction_time
    GRACE_PERIOD = "GRACE_PERIOD"  # allow data up to N seconds late
    LATEST_AVAILABLE = "LATEST_AVAILABLE"  # use whatever is available (no PIT guarantee)


@dataclass(frozen=True)
class FeatureDefinition:
    """Specification for a single feature within a FeatureView."""

    name: str
    description: str
    lookback_window: int  # number of periods
    aggregation: str  # "mean", "std", "ratio", "delta", etc.
    input_field: str  # which raw field this derives from
    min_periods: int = 1  # minimum data points needed (else None)


@dataclass(frozen=True)
class FeatureView:
    """Declarative specification of how features are computed from raw data.

    This is the contract between the feature engineering pipeline and
    downstream model training. Versioned, reproducible, auditable.
    """

    name: str
    version: str
    description: str
    entity_keys: tuple[str, ...]  # e.g., ("instrument_id",)
    input_sources: dict[str, str]  # {"trades": "canonical_trades_v2", "quotes": "canonical_quotes_v1"}
    lookback_window: int  # maximum lookback needed across all features
    features: tuple[FeatureDefinition, ...]
    point_in_time_rule: PointInTimeRule
    grace_period_seconds: int = 0  # only used if rule is GRACE_PERIOD
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

    @property
    def feature_names(self) -> tuple[str, ...]:
        return tuple(f.name for f in self.features)

    @property
    def max_lookback(self) -> int:
        return max(f.lookback_window for f in self.features)


# Define a FeatureView for a momentum strategy
momentum_view = FeatureView(
    name="fx_momentum_features",
    version="1.0.0",
    description="Momentum and volatility features for FX pairs, 20-day lookback, strict PIT",
    entity_keys=("instrument_id",),
    input_sources={
        "trades": "canonical_trades_v2",
        "quotes": "canonical_quotes_v1",
    },
    lookback_window=20,
    features=(
        FeatureDefinition(
            name="return_1d",
            description="Single-period percentage return",
            lookback_window=1,
            aggregation="delta",
            input_field="close_price",
        ),
        FeatureDefinition(
            name="volatility_20d",
            description="Rolling 20-day standard deviation of daily returns",
            lookback_window=20,
            aggregation="std",
            input_field="return_1d",
            min_periods=20,
        ),
        FeatureDefinition(
            name="momentum_20d",
            description="Ratio of positive return magnitude to total return magnitude over 20 days",
            lookback_window=20,
            aggregation="ratio",
            input_field="return_1d",
            min_periods=20,
        ),
        FeatureDefinition(
            name="volume_ratio_20d",
            description="Current volume relative to 20-day trailing average",
            lookback_window=20,
            aggregation="ratio",
            input_field="volume",
            min_periods=20,
        ),
        FeatureDefinition(
            name="price_vs_ma_20d",
            description="Price deviation from 20-day trailing moving average",
            lookback_window=20,
            aggregation="mean",
            input_field="close_price",
            min_periods=20,
        ),
    ),
    point_in_time_rule=PointInTimeRule.STRICT,
)

print("=== FeatureView Definition ===")
print(f"  Name:              {momentum_view.name}")
print(f"  Version:           {momentum_view.version}")
print(f"  Entity keys:       {momentum_view.entity_keys}")
print(f"  PIT rule:          {momentum_view.point_in_time_rule.value}")
print(f"  Max lookback:      {momentum_view.max_lookback} periods")
print(f"  Input sources:     {momentum_view.input_sources}")
print(f"  Features ({len(momentum_view.features)}):")
for feat in momentum_view.features:
    print(f"    - {feat.name}: {feat.description}")
    print(f"      lookback={feat.lookback_window}, agg={feat.aggregation}, input={feat.input_field}")
print()

# Serialize to JSON for storage/versioning
view_dict = {
    "name": momentum_view.name,
    "version": momentum_view.version,
    "entity_keys": list(momentum_view.entity_keys),
    "input_sources": momentum_view.input_sources,
    "point_in_time_rule": momentum_view.point_in_time_rule.value,
    "features": [
        {"name": f.name, "lookback": f.lookback_window, "aggregation": f.aggregation, "input": f.input_field}
        for f in momentum_view.features
    ],
}
print("=== Serialized for Version Control ===")
print(json.dumps(view_dict, indent=2))

# Try next:
# 1. Define a FeatureView for a mean-reversion strategy (spread, z-score features).
# 2. Add a GRACE_PERIOD FeatureView and explain when you would use it.
# 3. Write a validator that checks a FeatureView has no features with lookback > the view's lookback_window.

## 10. Experiment Tracking

Experiment tracking closes the loop from data to model to production. When a model in production starts making bad predictions, you need to trace back: what data was it trained on? What features? What hyperparameters? What code version? Without this traceability chain, debugging is guesswork.

### ExperimentRun

An ExperimentRun records everything about a single training run:

- **run_id**: unique identifier
- **dataset_snapshot_id**: which frozen dataset was used (links to DatasetSnapshot)
- **feature_view_version**: which feature definitions (links to FeatureView)
- **model_type**: the algorithm (linear regression, XGBoost, etc.)
- **parameters**: hyperparameters used
- **metrics**: evaluation results (accuracy, Sharpe ratio, etc.)
- **code_version**: git commit hash of the training code

### ModelArtifact

A ModelArtifact is the output of a training run — the trained model itself, tied back to the experiment that produced it.

- **artifact_id**: unique identifier
- **run_id**: which experiment produced this (links to ExperimentRun)
- **model_type and framework**: what it is
- **artifact_path**: where the serialized model lives
- **promoted_to**: deployment stage (staging, production, retired)

### The traceability chain

```
Production Model -> ModelArtifact -> ExperimentRun -> DatasetSnapshot -> FeatureView -> Raw Data
```

Every link in this chain must be preserved. If any link is broken, you cannot reproduce or debug the model.

### Tools context

MLflow, Weights & Biases, and cloud-native tools (SageMaker Experiments, Vertex AI) all implement this pattern. The data structures below show what these tools track under the hood.

In [ ]:
class DeploymentStage(Enum):
    DEVELOPMENT = "DEVELOPMENT"
    STAGING = "STAGING"
    PRODUCTION = "PRODUCTION"
    RETIRED = "RETIRED"


@dataclass(frozen=True)
class ExperimentRun:
    """Records everything about a single model training run."""

    run_id: str
    experiment_name: str
    dataset_snapshot_id: str  # FK to DatasetSnapshot
    feature_view_name: str
    feature_view_version: str
    model_type: str  # "linear_regression", "xgboost", "lstm", etc.
    parameters: dict[str, str | int | float]  # hyperparameters
    metrics: dict[str, float]  # evaluation results
    code_version: str  # git commit hash
    run_start: datetime
    run_end: datetime
    notes: str = ""

    @property
    def duration_seconds(self) -> float:
        return (self.run_end - self.run_start).total_seconds()


@dataclass(frozen=True)
class ModelArtifact:
    """A trained model artifact tied back to the experiment that produced it."""

    artifact_id: str
    run_id: str  # FK to ExperimentRun
    model_type: str
    framework: str  # "scikit-learn", "pytorch", "xgboost"
    artifact_path: str  # where the serialized model lives
    created_at: datetime
    promoted_to: DeploymentStage = DeploymentStage.DEVELOPMENT
    description: str = ""


# Simulate an experiment run
run = ExperimentRun(
    run_id="run-20260404-001",
    experiment_name="fx_momentum_v1",
    dataset_snapshot_id=snapshot.snapshot_id,
    feature_view_name=momentum_view.name,
    feature_view_version=momentum_view.version,
    model_type="logistic_regression",
    parameters={
        "C": 1.0,
        "penalty": "l2",
        "max_iter": 1000,
        "lookback": 20,
        "train_test_gap_days": 5,
    },
    metrics={
        "accuracy": 0.523,
        "precision": 0.518,
        "recall": 0.534,
        "f1": 0.526,
        "sharpe_ratio": 0.42,
        "max_drawdown": -0.034,
    },
    code_version="abc123def456",
    run_start=datetime(2026, 4, 4, 14, 0, 0, tzinfo=timezone.utc),
    run_end=datetime(2026, 4, 4, 14, 2, 30, tzinfo=timezone.utc),
    notes="Baseline logistic regression on 20-day momentum features",
)

artifact = ModelArtifact(
    artifact_id="model-20260404-001",
    run_id=run.run_id,
    model_type=run.model_type,
    framework="scikit-learn",
    artifact_path="s3://models/fx_momentum/run-20260404-001/model.pkl",
    created_at=run.run_end,
    promoted_to=DeploymentStage.STAGING,
    description="Baseline momentum classifier for EURUSD",
)

print("=== Experiment Run ===")
print(f"  run_id:          {run.run_id}")
print(f"  experiment:      {run.experiment_name}")
print(f"  dataset:         {run.dataset_snapshot_id}")
print(f"  feature_view:    {run.feature_view_name} v{run.feature_view_version}")
print(f"  model:           {run.model_type}")
print(f"  duration:        {run.duration_seconds:.0f}s")
print(f"  Parameters:")
for k, v in run.parameters.items():
    print(f"    {k}: {v}")
print(f"  Metrics:")
for k, v in run.metrics.items():
    print(f"    {k}: {v}")
print()

print("=== Model Artifact ===")
print(f"  artifact_id:     {artifact.artifact_id}")
print(f"  run_id:          {artifact.run_id}")
print(f"  framework:       {artifact.framework}")
print(f"  path:            {artifact.artifact_path}")
print(f"  stage:           {artifact.promoted_to.value}")
print()

# Full traceability chain
print("=== Traceability Chain ===")
print(f"  Model:     {artifact.artifact_id} ({artifact.promoted_to.value})")
print(f"    -> Run:      {run.run_id} ({run.model_type})")
print(f"    -> Dataset:  {snapshot.snapshot_id} ({snapshot.row_count} rows)")
print(f"    -> Features: {momentum_view.name} v{momentum_view.version}")
print(f"    -> Sources:  {snapshot.source_versions}")
print(f"    -> Code:     {run.code_version}")

# Try next:
# 1. Create a second run with different hyperparameters and compare metrics.
# 2. Promote the artifact to PRODUCTION and create a new one for STAGING.
# 3. Implement an ExperimentRegistry that tracks all runs and finds the best by a metric.

## 11. DataQualityReport

Data quality checks are not optional in trading — they are the immune system of your pipeline. Bad data that passes unchecked will produce bad features, bad models, and eventually bad trading decisions. A structured DataQualityReport captures the results of validation checks in a machine-readable format that can trigger alerts, block pipeline stages, and provide audit trails.

### What a quality report captures

- **Check name**: what was validated (e.g., "price_non_negative", "timestamp_monotonic")
- **Severity**: how bad is a failure (CRITICAL blocks the pipeline, WARNING logs and continues, INFO is informational)
- **Status**: PASS, FAIL, or SKIP
- **Affected scope**: which records or time range failed
- **Details**: human-readable explanation

### Common checks for trading data

| Check | What it catches |
|-------|----------------|
| Price > 0 | Corrupt or placeholder prices |
| Size > 0 | Zero-volume phantom trades |
| ask > bid | Crossed market (data corruption or stale quote) |
| Timestamp in expected range | Future timestamps, epoch-zero defaults |
| No duplicate trade_ids | Double-counted trades inflate VWAP |
| Monotonic timestamps | Out-of-order data (may be acceptable, depends on source) |
| Expected instruments present | Missing symbols = coverage gap |
| Price within N std of recent mean | Extreme outliers (fat-finger trades or bad data) |

In [ ]:
class Severity(Enum):
    CRITICAL = "CRITICAL"  # blocks pipeline
    WARNING = "WARNING"  # logs and continues
    INFO = "INFO"  # informational


class CheckStatus(Enum):
    PASS = "PASS"
    FAIL = "FAIL"
    SKIP = "SKIP"


@dataclass(frozen=True)
class QualityCheck:
    """Result of a single data quality check."""

    check_name: str
    severity: Severity
    status: CheckStatus
    affected_count: int
    total_count: int
    details: str

    @property
    def pass_rate(self) -> float:
        if self.total_count == 0:
            return 1.0
        return (self.total_count - self.affected_count) / self.total_count


@dataclass
class DataQualityReport:
    """Structured result of all quality checks on a data batch."""

    report_id: str
    source: str
    check_time: datetime
    data_start: datetime
    data_end: datetime
    record_count: int
    checks: list[QualityCheck] = field(default_factory=list)

    @property
    def passed(self) -> bool:
        """A report passes if no CRITICAL checks failed."""
        return not any(
            c.status == CheckStatus.FAIL and c.severity == Severity.CRITICAL
            for c in self.checks
        )

    @property
    def summary(self) -> dict[str, int]:
        return {
            "total_checks": len(self.checks),
            "passed": sum(1 for c in self.checks if c.status == CheckStatus.PASS),
            "failed": sum(1 for c in self.checks if c.status == CheckStatus.FAIL),
            "skipped": sum(1 for c in self.checks if c.status == CheckStatus.SKIP),
        }


def run_trade_quality_checks(trades: list[Trade]) -> DataQualityReport:
    """Run standard quality checks on a batch of trades."""
    checks: list[QualityCheck] = []
    n = len(trades)

    if n == 0:
        return DataQualityReport(
            report_id=f"dqr-{uuid.uuid4().hex[:8]}",
            source="unknown",
            check_time=datetime.now(timezone.utc),
            data_start=datetime.min.replace(tzinfo=timezone.utc),
            data_end=datetime.min.replace(tzinfo=timezone.utc),
            record_count=0,
            checks=[QualityCheck("non_empty_batch", Severity.CRITICAL, CheckStatus.FAIL, 0, 0, "Empty batch")],
        )

    # Check 1: Price > 0
    bad_prices = [t for t in trades if t.price <= 0]
    checks.append(QualityCheck(
        check_name="price_positive",
        severity=Severity.CRITICAL,
        status=CheckStatus.FAIL if bad_prices else CheckStatus.PASS,
        affected_count=len(bad_prices),
        total_count=n,
        details=f"{len(bad_prices)} trades with price <= 0" if bad_prices else "All prices positive",
    ))

    # Check 2: Size > 0
    bad_sizes = [t for t in trades if t.size <= 0]
    checks.append(QualityCheck(
        check_name="size_positive",
        severity=Severity.CRITICAL,
        status=CheckStatus.FAIL if bad_sizes else CheckStatus.PASS,
        affected_count=len(bad_sizes),
        total_count=n,
        details=f"{len(bad_sizes)} trades with size <= 0" if bad_sizes else "All sizes positive",
    ))

    # Check 3: No duplicate trade_ids
    trade_ids = [t.trade_id for t in trades]
    dup_count = len(trade_ids) - len(set(trade_ids))
    checks.append(QualityCheck(
        check_name="no_duplicate_trade_ids",
        severity=Severity.WARNING,
        status=CheckStatus.FAIL if dup_count > 0 else CheckStatus.PASS,
        affected_count=dup_count,
        total_count=n,
        details=f"{dup_count} duplicate trade_ids" if dup_count else "All trade_ids unique",
    ))

    # Check 4: event_time not in the future
    now = datetime.now(timezone.utc)
    future_trades = [t for t in trades if t.event_time > now]
    checks.append(QualityCheck(
        check_name="no_future_timestamps",
        severity=Severity.WARNING,
        status=CheckStatus.FAIL if future_trades else CheckStatus.PASS,
        affected_count=len(future_trades),
        total_count=n,
        details=f"{len(future_trades)} trades with future timestamps" if future_trades else "All timestamps valid",
    ))

    # Check 5: ingest_time >= event_time (causality)
    causal_violations = [t for t in trades if t.ingest_time < t.event_time]
    checks.append(QualityCheck(
        check_name="causal_timestamps",
        severity=Severity.WARNING,
        status=CheckStatus.FAIL if causal_violations else CheckStatus.PASS,
        affected_count=len(causal_violations),
        total_count=n,
        details=f"{len(causal_violations)} trades where ingest < event" if causal_violations else "All timestamps causal",
    ))

    sorted_trades = sorted(trades, key=lambda t: t.event_time)
    return DataQualityReport(
        report_id=f"dqr-{uuid.uuid4().hex[:8]}",
        source=trades[0].source,
        check_time=datetime.now(timezone.utc),
        data_start=sorted_trades[0].event_time,
        data_end=sorted_trades[-1].event_time,
        record_count=n,
        checks=checks,
    )


# Run quality checks on our sample trades
report = run_trade_quality_checks(sample_trades)

print("=== Data Quality Report ===")
print(f"  report_id:    {report.report_id}")
print(f"  source:       {report.source}")
print(f"  records:      {report.record_count}")
print(f"  period:       {report.data_start.strftime('%H:%M:%S')} to {report.data_end.strftime('%H:%M:%S')}")
print(f"  overall:      {'PASS' if report.passed else 'FAIL'}")
print(f"  summary:      {report.summary}")
print()
for check in report.checks:
    status_marker = "OK" if check.status == CheckStatus.PASS else "FAIL"
    print(f"  [{check.severity.value:8s}] {status_marker:4s} {check.check_name}: {check.details} ({check.pass_rate:.0%})")

# Try next:
# 1. Create a trade with price=0 and verify the quality check catches it.
# 2. Add a check for "price within 3 standard deviations of the mean" (outlier detection).
# 3. Implement a pipeline gate: if report.passed is False, raise an exception.

## 12. Vendor Mapping and Reconciliation

In production, you receive trade data from multiple vendors — each with their own schema, field names, data types, and quirks. The vendor mapping exercise normalizes all sources into your canonical Trade model, then reconciles to find discrepancies.

### Why reconciliation matters

- Vendor A might report a trade that Vendor B missed (coverage gap)
- Vendor A and B might report the same trade with slightly different prices (data quality issue)
- One vendor might be faster but less accurate; the other slower but more complete
- Regulatory reporting requires knowing which trades are "real" vs duplicates

### The exercise below

Two mock vendors send the same EURUSD trades but with completely different schemas:
- **Vendor Alpha**: JSON format, uses "px" for price, "qty" for quantity, Unix timestamps
- **Vendor Beta**: CSV-like dicts, uses "execution_price", "amount", ISO timestamps

We normalize both to the canonical Trade model, then reconcile by matching on (instrument, event_time, price, size).

In [ ]:
# === Mock vendor data ===

# Vendor Alpha: JSON-style, Unix timestamps, abbreviated field names
vendor_alpha_raw = [
    {"id": "A001", "sym": "EUR/USD", "px": 1.08450, "qty": 2000000, "side": "B", "ts": 1743760800.123, "venue": "EBS"},
    {"id": "A002", "sym": "EUR/USD", "px": 1.08455, "qty": 3000000, "side": "S", "ts": 1743760805.456, "venue": "EBS"},
    {"id": "A003", "sym": "EUR/USD", "px": 1.08440, "qty": 1500000, "side": "S", "ts": 1743760815.789, "venue": "EBS"},
    {"id": "A004", "sym": "EUR/USD", "px": 1.08460, "qty": 5000000, "side": "B", "ts": 1743760830.012, "venue": "EBS"},
    {"id": "A005", "sym": "EUR/USD", "px": 1.08470, "qty": 1000000, "side": "B", "ts": 1743760845.345, "venue": "EBS"},  # Alpha-only trade
]

# Vendor Beta: CSV-style, ISO timestamps, verbose field names
vendor_beta_raw = [
    {"trade_ref": "B-10001", "instrument": "EURUSD", "execution_price": "1.08450", "amount": "2000000", "direction": "BUY", "timestamp": "2025-04-04T10:00:00.123000+00:00", "exchange": "EBS"},
    {"trade_ref": "B-10002", "instrument": "EURUSD", "execution_price": "1.08455", "amount": "3000000", "direction": "SELL", "timestamp": "2025-04-04T10:00:05.456000+00:00", "exchange": "EBS"},
    {"trade_ref": "B-10003", "instrument": "EURUSD", "execution_price": "1.08440", "amount": "1500000", "direction": "SELL", "timestamp": "2025-04-04T10:00:15.789000+00:00", "exchange": "EBS"},
    {"trade_ref": "B-10004", "instrument": "EURUSD", "execution_price": "1.08460", "amount": "5000000", "direction": "BUY", "timestamp": "2025-04-04T10:00:30.012000+00:00", "exchange": "EBS"},
    {"trade_ref": "B-10005", "instrument": "EURUSD", "execution_price": "1.08442", "amount": "800000", "direction": "SELL", "timestamp": "2025-04-04T10:00:20.000000+00:00", "exchange": "EBS"},  # Beta-only trade
]


# === Normalizers: vendor-specific -> canonical Trade ===

def normalize_alpha(raw: dict) -> Trade:
    """Normalize Vendor Alpha's format to canonical Trade."""
    # Symbol mapping: Alpha uses "EUR/USD" with slash
    symbol = raw["sym"].replace("/", "")  # EUR/USD -> EURUSD
    instrument_id = mapper.resolve("EBS", symbol) or f"UNKNOWN-{symbol}"

    # Side mapping: Alpha uses single char
    side = Side.BUY if raw["side"] == "B" else Side.SELL

    # Timestamp: Alpha uses Unix epoch
    event_time = datetime.fromtimestamp(raw["ts"], tz=timezone.utc)
    ingest_time = datetime.now(timezone.utc)

    return Trade(
        source="VENDOR_ALPHA",
        trade_id=raw["id"],
        instrument_id=instrument_id,
        event_time=event_time,
        ingest_time=ingest_time,
        price=Decimal(str(raw["px"])),
        size=Decimal(str(raw["qty"])),
        side=side,
    )


def normalize_beta(raw: dict) -> Trade:
    """Normalize Vendor Beta's format to canonical Trade."""
    # Symbol mapping: Beta uses "EURUSD" (no separator)
    instrument_id = mapper.resolve("EBS", raw["instrument"]) or f"UNKNOWN-{raw['instrument']}"

    # Side mapping: Beta uses full word
    side = Side.BUY if raw["direction"] == "BUY" else Side.SELL

    # Timestamp: Beta uses ISO format
    event_time = datetime.fromisoformat(raw["timestamp"])
    ingest_time = datetime.now(timezone.utc)

    return Trade(
        source="VENDOR_BETA",
        trade_id=raw["trade_ref"],
        instrument_id=instrument_id,
        event_time=event_time,
        ingest_time=ingest_time,
        price=Decimal(raw["execution_price"]),
        size=Decimal(raw["amount"]),
        side=side,
    )


# Normalize both vendors
alpha_trades = [normalize_alpha(r) for r in vendor_alpha_raw]
beta_trades = [normalize_beta(r) for r in vendor_beta_raw]

print("=== Normalized Trades ===")
print(f"  Alpha: {len(alpha_trades)} trades")
print(f"  Beta:  {len(beta_trades)} trades")
print()


# === Reconciliation ===

def reconciliation_key(t: Trade) -> tuple:
    """Create a matching key for reconciliation.

    We match on (instrument, event_time rounded to nearest second, price, size).
    Time rounding accounts for microsecond-level timestamp differences between vendors.
    """
    rounded_time = t.event_time.replace(microsecond=0)
    return (t.instrument_id, rounded_time, t.price, t.size)


alpha_keys = {reconciliation_key(t): t for t in alpha_trades}
beta_keys = {reconciliation_key(t): t for t in beta_trades}

matched_keys = set(alpha_keys.keys()) & set(beta_keys.keys())
alpha_only = set(alpha_keys.keys()) - set(beta_keys.keys())
beta_only = set(beta_keys.keys()) - set(alpha_keys.keys())

print("=== Reconciliation Results ===")
print(f"  Matched:    {len(matched_keys)} trades found in both vendors")
print(f"  Alpha-only: {len(alpha_only)} trades only in Vendor Alpha")
print(f"  Beta-only:  {len(beta_only)} trades only in Vendor Beta")
print()

if alpha_only:
    print("  Alpha-only trades (not in Beta):")
    for key in alpha_only:
        t = alpha_keys[key]
        print(f"    {t.trade_id}: {t.price} x {t.size:,} @ {t.event_time.strftime('%H:%M:%S')}")

if beta_only:
    print("  Beta-only trades (not in Alpha):")
    for key in beta_only:
        t = beta_keys[key]
        print(f"    {t.trade_id}: {t.price} x {t.size:,} @ {t.event_time.strftime('%H:%M:%S')}")

print()
print("  Matched trades (showing both vendor IDs):")
for key in sorted(matched_keys, key=lambda k: k[1]):
    a = alpha_keys[key]
    b = beta_keys[key]
    print(f"    Alpha={a.trade_id} <-> Beta={b.trade_id}: {a.price} x {a.size:,}")

# Try next:
# 1. What if Alpha reports a price of 1.08451 and Beta reports 1.08450? (price tolerance)
# 2. Implement a "reconciliation rate" metric: matched / total unique trades.
# 3. Build a function that merges the reconciled data into a single deduplicated stream.

---
## Part II: Data Decisions and Their Downstream Effects

Every decision below propagates through the entire pipeline. The goal is not to memorize rules
but to develop intuition for *why* a data choice produces a specific downstream effect.

### Sections
- A. Price representation and stationarity
- B. Gap filling strategies
- C. Outlier handling
- D. Normalization — the most common hidden leakage source
- E. Leakage taxonomy — all the forms it takes
- F. Label construction choices
- G. Train/validation strategies — walk-forward vs. random split
- H. Feature window size — noise vs. lag trade-off
- I. Class imbalance and evaluation metrics
- J. Principles summary (interview-ready)

All demonstrations use Python standard library only (`random`, `math`, `statistics`).
Every effect is observable by changing one parameter and re-running.


### A. Price Representation and Stationarity

**The problem with raw prices:** prices are non-stationary — their mean and variance drift over time.
A model trained on 2020 prices (range 50–150) sees completely different values in 2025 (range 200–400).
The numeric relationship it learned is an artifact of the price *level*, not the signal.

**Returns:** `r_t = (P_t - P_{t-1}) / P_{t-1}` — percentage change. Approximately stationary.

**Log-returns:** `lr_t = ln(P_t / P_{t-1})` — preferred for three reasons:
1. **Additive across time:** the 5-day log-return equals the sum of 5 daily log-returns (exact identity)
2. **Symmetric:** a +10% day followed by a -10% day is NOT zero in raw returns (−1%), but nearly zero in log-returns
3. **Closer to Gaussian:** most risk models and option pricing assume log-normality

**Key rule:** never use raw price as a model feature. Transform to returns or log-returns first.
Stationarity check: plot rolling mean and rolling std. For raw price both drift; for returns both stay roughly constant.


In [ ]:
import random
import math
import statistics


def generate_gbm_prices(
    n: int = 300,
    start: float = 100.0,
    drift: float = 0.0003,
    vol: float = 0.015,
    seed: int = 42,
) -> list[float]:
    """Geometric Brownian Motion — standard equity price simulation."""
    random.seed(seed)
    prices = [start]
    for _ in range(n - 1):
        shock = random.gauss(0, 1)
        log_ret = drift - 0.5 * vol ** 2 + vol * shock  # Ito correction
        prices.append(prices[-1] * math.exp(log_ret))
    return prices


prices = generate_gbm_prices(n=300)
raw_returns = [(prices[i] - prices[i - 1]) / prices[i - 1] for i in range(1, len(prices))]
log_returns = [math.log(prices[i] / prices[i - 1]) for i in range(1, len(prices))]

# Demonstrate log-return additivity: sum of daily = cumulative (exact)
lr_sum_5 = sum(log_returns[0:5])
lr_direct_5 = math.log(prices[5] / prices[0])

# Raw returns are NOT additive: compound product ≠ simple sum
rr_compound_5 = (
    (1 + raw_returns[0]) * (1 + raw_returns[1]) * (1 + raw_returns[2]) *
    (1 + raw_returns[3]) * (1 + raw_returns[4]) - 1
)
rr_direct_5 = (prices[5] - prices[0]) / prices[0]

# Rolling mean: price drifts; returns stay near zero
W = 60
rolling_price_means = [statistics.mean(prices[i - W:i]) for i in range(W, len(prices))]
rolling_return_means = [statistics.mean(raw_returns[i - W:i]) for i in range(W, len(raw_returns))]


def summarize(values, name):
    return {
        "series": name,
        "mean": round(statistics.mean(values), 6),
        "std": round(statistics.stdev(values), 6),
        "min": round(min(values), 4),
        "max": round(max(values), 4),
    }


{
    "summaries": [
        summarize(prices, "raw_price"),
        summarize(raw_returns, "raw_return"),
        summarize(log_returns, "log_return"),
    ],
    "log_return_additivity": {
        "sum_of_5_daily": round(lr_sum_5, 10),
        "5_day_direct": round(lr_direct_5, 10),
        "exact_match": abs(lr_sum_5 - lr_direct_5) < 1e-10,
    },
    "raw_return_not_additive": {
        "compound_product": round(rr_compound_5, 8),
        "5_day_direct": round(rr_direct_5, 8),
        "they_differ": abs(rr_compound_5 - rr_direct_5) > 1e-10,
    },
    "stationarity_check": {
        "rolling_price_mean_range": [round(min(rolling_price_means), 2), round(max(rolling_price_means), 2)],
        "rolling_return_mean_range": [round(min(rolling_return_means), 6), round(max(rolling_return_means), 6)],
        "conclusion": "price mean drifts widely; return mean stays near zero (stationary-like)",
    },
}


### B. Gap Filling Strategies

Real market data has gaps: pre-market bars, holidays, illiquid instruments, vendor outages.
How you fill those gaps propagates directly into every downstream feature.

| Strategy | How it works | Use when |
| --- | --- | --- |
| **Forward fill (LOCF)** | Repeat the last known value | Price — last traded price is still "valid" |
| **Zero fill** | Insert 0 for missing | Volume — no activity = zero activity |
| **Linear interpolation** | Linearly connect surrounding known points | Slowly-changing reference data |
| **Leave as None** | Mark missing, handle downstream | When "no data" is itself meaningful |

**Critical insight:** forward-fill creates runs of identical prices → zero returns in those windows →
rolling volatility is artificially *suppressed*.
Zero-fill creates massive spike-and-return artifacts → rolling vol is artificially *inflated*.

Try: change `GAP_PROBABILITY` from 0.05 to 0.20 and observe how vol statistics shift.


In [ ]:
import random
import math
import statistics


GAP_PROBABILITY = 0.07  # <-- change and re-run to observe the effect


def introduce_gaps(series, prob, seed=99):
    random.seed(seed)
    return [None if (i > 0 and random.random() < prob) else v for i, v in enumerate(series)]


def forward_fill(series):
    last, result = None, []
    for v in series:
        last = v if v is not None else last
        result.append(last)
    return result


def zero_fill(series):
    return [0.0 if v is None else v for v in series]


def linear_interpolate(series):
    result = list(series)
    i = 0
    while i < len(result):
        if result[i] is None:
            j = i + 1
            while j < len(result) and result[j] is None:
                j += 1
            if j < len(result) and i > 0:
                start_v, end_v = result[i - 1], result[j]
                gap_len = j - i + 1
                for k in range(i, j):
                    t = (k - i + 1) / gap_len
                    result[k] = start_v + t * (end_v - start_v)
            i = j
        else:
            i += 1
    return result


def rolling_annualized_vol(series, window=20):
    """Rolling std of 1-period returns, annualized."""
    rets = []
    for i in range(1, len(series)):
        if series[i] is not None and series[i - 1] is not None and series[i - 1] != 0:
            rets.append((series[i] - series[i - 1]) / series[i - 1])
        else:
            rets.append(None)
    result = []
    for i in range(window, len(rets)):
        w = [r for r in rets[i - window:i] if r is not None]
        result.append(statistics.stdev(w) * math.sqrt(252) if len(w) >= window // 2 else None)
    return result


prices_clean = generate_gbm_prices(n=200)
prices_gapped = introduce_gaps(prices_clean, GAP_PROBABILITY)
gap_count = sum(1 for v in prices_gapped if v is None)

series = {
    "clean": prices_clean,
    "forward_fill": forward_fill(prices_gapped),
    "zero_fill": zero_fill(prices_gapped),
    "interpolate": linear_interpolate(prices_gapped),
}


def vol_summary(name, series_data):
    vols = [v for v in rolling_annualized_vol(series_data) if v is not None]
    if not vols:
        return {"strategy": name, "error": "no valid vols"}
    return {
        "strategy": name,
        "mean_vol": round(statistics.mean(vols), 4),
        "std_of_vol": round(statistics.stdev(vols), 4),
        "max_vol": round(max(vols), 4),
    }


{
    "gap_count": gap_count,
    "gap_rate": f"{gap_count / len(prices_gapped):.1%}",
    "vol_comparison": [vol_summary(k, v) for k, v in series.items()],
    "insight": {
        "forward_fill": "suppresses vol — identical prices in gap windows produce zero returns",
        "zero_fill": "massively inflates vol — every gap creates a price-to-zero spike",
        "interpolation": "closest to ground truth for smooth price series",
        "recommendation": "use LOCF for prices; zero-fill for volume; NaN for illiquid instruments",
    },
}


### C. Outlier Handling

Two kinds of outliers in market data — they need *different* treatment:

1. **Data errors** (bad ticks, fat-finger prints, decimal place errors): should be cleaned
2. **Real extreme events** (flash crashes, COVID, Black Monday): part of the true distribution —
   removing them teaches the model a false world where large moves never happen

**Four strategies:**

| Strategy | What it does | Best for |
| --- | --- | --- |
| **Winsorization** | Clip to [p₁, p₉₉] quantile bounds | Robust feature normalization while keeping extremes |
| **Rolling Z-score clip** | Replace points > N·σ from rolling mean | Bad ticks and point anomalies |
| **Log transform** | `sign(r) · log(1 + |r|)` compresses large values | Right-skewed data (volume, spread, ADV) |
| **No treatment** | Accept the tail | Volatility modelling where extremes are the signal |

**Key insight:** outlier treatment changes rolling statistics, which changes every feature built on those statistics.
If you winsorize returns at ±3%, your rolling-vol and momentum features will both be capped indirectly.

Try changing `OUTLIER_MAGNITUDE_SIGMA` from 6 to 12 and re-run.


In [ ]:
import random
import math
import statistics


N_OUTLIERS = 5              # <-- change and re-run
OUTLIER_MAGNITUDE_SIGMA = 8  # <-- how many σ the injected outlier is


def inject_outliers(returns, n, magnitude_sigma, seed=7):
    random.seed(seed)
    ret_std = statistics.stdev(returns)
    dirty = list(returns)
    indices = random.sample(range(10, len(returns) - 10), n)
    for idx in indices:
        direction = random.choice([-1, 1])
        dirty[idx] = direction * magnitude_sigma * ret_std
    return dirty, indices


def winsorize(values, lower_pct=1.0, upper_pct=99.0):
    n = len(values)
    sv = sorted(values)
    lo = sv[max(0, int(n * lower_pct / 100))]
    hi = sv[min(n - 1, int(n * upper_pct / 100))]
    return [max(lo, min(hi, v)) for v in values]


def rolling_zscore_clip(values, window=30, threshold=3.5):
    result = list(values)
    for i in range(window, len(values)):
        w = values[i - window:i]
        mu = statistics.mean(w)
        sigma = statistics.stdev(w) if len(w) > 1 else 0
        if sigma > 0 and abs((values[i] - mu) / sigma) > threshold:
            result[i] = mu
    return result


def log_transform(returns):
    return [math.copysign(math.log(1 + abs(r)), r) for r in returns]


def rolling_vol_from_returns(returns, window=20):
    result = []
    for i in range(window, len(returns)):
        result.append(statistics.stdev(returns[i - window:i]) * math.sqrt(252))
    return result


prices_c = generate_gbm_prices(n=200)
base_returns = [(prices_c[i] - prices_c[i - 1]) / prices_c[i - 1] for i in range(1, len(prices_c))]
dirty_returns, outlier_idx = inject_outliers(base_returns, N_OUTLIERS, OUTLIER_MAGNITUDE_SIGMA)

strategies = {
    "clean_baseline": base_returns,
    "dirty_no_treatment": dirty_returns,
    "winsorized_1_99": winsorize(dirty_returns, 1, 99),
    "rolling_zscore_clipped": rolling_zscore_clip(dirty_returns),
    "log_transformed": log_transform(dirty_returns),
}


def feature_stats(returns, name):
    vols = rolling_vol_from_returns(returns)
    return {
        "strategy": name,
        "return_std": round(statistics.stdev(returns), 5),
        "max_abs_return": round(max(abs(r) for r in returns), 4),
        "mean_rolling_vol": round(statistics.mean(vols), 4),
        "max_rolling_vol": round(max(vols), 4),
    }


{
    "injected_outlier_indices": outlier_idx[:3],
    "feature_stats": [feature_stats(v, k) for k, v in strategies.items()],
    "insight": {
        "dirty_no_treatment": "outliers inflate rolling vol — every window containing one is wrong",
        "winsorized": "caps extremes but windows near outliers still show elevated vol",
        "rolling_zscore_clip": "best for point anomalies; local mean replaced — distribution restored",
        "log_transform": "compresses tails gently; preserves sign and ordering",
        "danger": "removing real market crashes teaches the model that crashes are impossible",
    },
}


### D. Normalization Strategies — The Hidden Leakage Trap

Normalization is where many practitioners unknowingly introduce look-ahead bias.

**The most common mistake:** apply global z-score normalization *before* splitting into train/test.
The mean and std used to normalize your TRAINING data were computed from FUTURE data in your TEST set.
The model implicitly "knows" the future statistical range.

| Strategy | Uses future? | Notes |
| --- | --- | --- |
| **Global z-score** | YES — leaky | Never use for time-series |
| **Global min-max** | YES — leaky | Min/max include future extremes |
| **Rolling z-score** | No — safe | Normalize at t using only past W observations |
| **Rank normalization** | No — safe | Percentile rank, robust to outliers |

**Why rolling z-score is the right default:**
At each time step t: `z_t = (x_t - mean(x_{t-W}, …, x_{t-1})) / std(x_{t-W}, …, x_{t-1})`
Only past data. The normalized value at t is point-in-time correct.

**Secondary issue:** non-stationarity in variance (GARCH effects).
Volatility clusters. A global std blends high-vol and low-vol regimes, producing features with
different meaning in each. Rolling normalization adapts to the current regime.


In [ ]:
import statistics
import math


def global_zscore(values):
    mu = statistics.mean(values)
    sigma = statistics.stdev(values)
    return [(v - mu) / sigma for v in values], mu, sigma


def rolling_zscore(values, window):
    """Walk-forward: normalize at t using only data from [t-window, t-1]."""
    result = [None] * window
    for i in range(window, len(values)):
        w = values[i - window:i]
        mu = statistics.mean(w)
        sigma = statistics.stdev(w) if len(w) > 1 else 1.0
        result.append((values[i] - mu) / sigma if sigma > 0 else 0.0)
    return result


def rank_normalize(values):
    """Percentile rank — robust to outliers, regime-stable."""
    n = len(values)
    ranked = sorted(range(n), key=lambda i: values[i])
    result = [0.0] * n
    for rank, idx in enumerate(ranked):
        result[idx] = rank / (n - 1)
    return result


# Build a raw feature: 20-day rolling volatility
prices_d = generate_gbm_prices(n=400)
returns_d = [(prices_d[i] - prices_d[i - 1]) / prices_d[i - 1] for i in range(1, len(prices_d))]

VOL_WINDOW = 20
raw_feature = [
    statistics.stdev(returns_d[i - VOL_WINDOW:i]) * math.sqrt(252)
    for i in range(VOL_WINDOW, len(returns_d))
]

# Apply normalization strategies
global_z, global_mu, global_sigma = global_zscore(raw_feature)
rolling_z = rolling_zscore(raw_feature, window=60)
rank_n = rank_normalize(raw_feature)

# Demonstrate leakage: split at 80%
split = int(0.8 * len(raw_feature))
train_feat, test_feat = raw_feature[:split], raw_feature[split:]

train_mu = statistics.mean(train_feat)
train_sigma = statistics.stdev(train_feat)
test_z_from_train = [(v - train_mu) / train_sigma for v in test_feat]
test_z_global = global_z[split:]  # uses full-sample stats — leaky

{
    "leakage_evidence": {
        "full_sample_mu": round(global_mu, 6),
        "train_only_mu": round(train_mu, 6),
        "full_sample_sigma": round(global_sigma, 6),
        "train_only_sigma": round(train_sigma, 6),
        "mu_contamination_pct": round(abs(global_mu - train_mu) / train_mu * 100, 2),
        "sigma_contamination_pct": round(abs(global_sigma - train_sigma) / train_sigma * 100, 2),
    },
    "test_period_range_comparison": {
        "global_z_range": [round(min(test_z_global), 3), round(max(test_z_global), 3)],
        "train_only_z_range": [round(min(test_z_from_train), 3), round(max(test_z_from_train), 3)],
        "note": "global z artificially compresses test extremes because global sigma is larger",
    },
    "rolling_z_valid_points": sum(1 for v in rolling_z if v is not None),
    "rule": "always normalize using only data available at prediction time — rolling window only",
}


### E. Leakage Taxonomy — All the Forms It Takes

Leakage = any information from outside the valid knowledge boundary entering the model's training process.
In trading the boundary is simple: **at time t, you only know what was observable at or before time t.**

| Type | Description | Fix |
| --- | --- | --- |
| **Split-time** | Random train/test split — train rows are adjacent to test rows (autocorrelated) | Always split on time |
| **Feature-time** | Feature at t uses global/future statistics | Rolling/expanding window features only |
| **Label** | Label construction uses same-bar or future data | Ensure feature and label windows don't overlap |
| **Survivorship** | Only modeling instruments that still exist today | Include delisted instruments in the universe |
| **Reference data** | Retroactively corrected prices/fundamentals (restated earnings, split adjustments) | Use bitemporal/as-of data (Section 4 of this notebook) |

**The measurable symptom:** a large gap between in-sample and out-of-sample performance.
A perfectly fit in-sample model with poor out-of-sample performance is almost always leakage or overfitting.

**Autocorrelation and split-time leakage:**
If your feature uses a 20-day window, rows within 20 days of each other share input data.
A random split puts row t in train and row t+5 in test — they share 15 days of inputs.
The model indirectly learns the test rows during training.


In [ ]:
import random
import statistics


def build_toy_trending_dataset(n=400, seed=42):
    """
    Dataset with a real but weak signal:
    if 20-day mean return > 0, next-bar is slightly more likely to be up.
    """
    random.seed(seed)
    prices = generate_gbm_prices(n=n + 22, drift=0.0004, vol=0.015)
    returns = [(prices[i] - prices[i - 1]) / prices[i - 1] for i in range(1, len(prices))]
    rows = []
    for i in range(20, n + 20):
        feat = statistics.mean(returns[i - 20:i])
        label = 1 if returns[i] > 0 else 0
        rows.append((feat, label))
    return rows


def best_threshold(train_rows):
    thresholds = sorted(set(f for f, _ in train_rows))
    best_t, best_acc = thresholds[0], 0.0
    for t in thresholds:
        preds = [1 if f > t else 0 for f, _ in train_rows]
        acc = sum(p == l for p, (_, l) in zip(preds, train_rows)) / len(train_rows)
        if acc > best_acc:
            best_acc, best_t = acc, t
    return best_t, best_acc


def evaluate_threshold(test_rows, threshold):
    preds = [1 if f > threshold else 0 for f, _ in test_rows]
    acc = sum(p == l for p, (_, l) in zip(preds, test_rows)) / len(test_rows)
    return acc


rows = build_toy_trending_dataset(n=400)
n = len(rows)

# --- Strategy 1: Random split (LEAKY) ---
shuffled = list(rows)
random.seed(0)
random.shuffle(shuffled)
split = int(0.8 * n)
train_r, test_r = shuffled[:split], shuffled[split:]
t_rand, train_acc_rand = best_threshold(train_r)
test_acc_rand = evaluate_threshold(test_r, t_rand)

# --- Strategy 2: Time-based split (CORRECT) ---
train_t, test_t = rows[:split], rows[split:]
t_time, train_acc_time = best_threshold(train_t)
test_acc_time = evaluate_threshold(test_t, t_time)

# --- Baseline: always predict majority ---
test_labels = [l for _, l in test_t]
majority = 1 if sum(test_labels) > len(test_labels) / 2 else 0
baseline = sum(l == majority for l in test_labels) / len(test_labels)

{
    "random_split": {
        "train_acc": round(train_acc_rand, 3),
        "test_acc": round(test_acc_rand, 3),
        "note": "inflated — autocorrelated rows leak across split boundary",
    },
    "time_based_split": {
        "train_acc": round(train_acc_time, 3),
        "test_acc": round(test_acc_time, 3),
        "note": "realistic — train set never sees future rows",
    },
    "always_majority_baseline": round(baseline, 3),
    "insight": (
        "random split test accuracy is artificially high. "
        "The model learns test-period context through autocorrelated features "
        "that happen to share overlapping windows with nearby test rows."
    ),
}


### F. Label Construction Choices

The label you construct determines what the model is actually trying to learn.
Changing only the label changes signal quality, class balance, and model usefulness —
even when features and data are identical.

**Four dimensions to choose across:**

**1. Horizon:** how far ahead do you predict?
- 1-bar: maximum noise, minimum autocorrelation — the hardest prediction problem
- 5-bar: smooths day-of-week effects, slightly higher autocorrelation
- 20-bar: captures short-term trends but features and labels start to overlap (leakage risk)

**2. Label type**
- Binary (up/down): simplest, loses magnitude information
- Tertile (top/middle/bottom third): adds a "flat" class that reduces false directional calls
- Regression (predict the return value): uses full information, but requires different models

**3. Volatility adjustment (Sharpe-style label)**
- `label = forward_return / rolling_vol`
- Normalizes by the current risk regime — a 0.5% move is different in low-vol vs. high-vol periods
- Creates regime-consistent labels that are more comparable across market conditions

**4. Threshold design**
- Symmetric (0%): maximum class count, maximum noise
- Economic threshold: only label as "up" if expected return > transaction cost estimate
- Adaptive threshold: scale with current volatility (avoids forcing binary labels on flat markets)


In [ ]:
import random
import math
import statistics


random.seed(42)
prices_f = generate_gbm_prices(n=500, drift=0.0003, vol=0.015)
returns_f = [(prices_f[i] - prices_f[i - 1]) / prices_f[i - 1] for i in range(1, len(prices_f))]


def binary_labels(returns, horizon):
    labels = []
    for i in range(len(returns) - horizon):
        fwd = sum(returns[i:i + horizon])
        labels.append(1 if fwd > 0 else 0)
    return labels + [None] * horizon


def sharpe_labels(returns, horizon=5, vol_window=20, threshold=0.3):
    """1 if forward Sharpe ratio > threshold, else 0."""
    labels = []
    for i in range(vol_window, len(returns) - horizon):
        rv = statistics.stdev(returns[i - vol_window:i])
        fwd_ret = sum(returns[i:i + horizon])
        sharpe = fwd_ret / (rv * math.sqrt(horizon)) if rv > 0 else 0
        labels.append(1 if sharpe > threshold else 0)
    return labels


def tertile_labels(returns, horizon=5):
    """Classify forward return into bottom / middle / top tertile."""
    fwd = [sum(returns[i:i + horizon]) for i in range(len(returns) - horizon)]
    sv = sorted(fwd)
    n = len(sv)
    lo = sv[n // 3]
    hi = sv[2 * n // 3]
    return ["DOWN" if r <= lo else ("UP" if r >= hi else "FLAT") for r in fwd]


def autocorr(labels, lag=1):
    """How often does label[t] == label[t+lag]? High = more predictable structure."""
    valid = [l for l in labels if l is not None and isinstance(l, int)]
    if len(valid) < lag + 1:
        return None
    matches = sum(valid[i] == valid[i + lag] for i in range(len(valid) - lag))
    return round(matches / (len(valid) - lag), 3)


labels_1d = [l for l in binary_labels(returns_f, 1) if l is not None]
labels_5d = [l for l in binary_labels(returns_f, 5) if l is not None]
labels_20d = [l for l in binary_labels(returns_f, 20) if l is not None]
labels_sharpe = sharpe_labels(returns_f, horizon=5, vol_window=20, threshold=0.3)
labels_tertile = tertile_labels(returns_f, horizon=5)

tertile_counts = {v: labels_tertile.count(v) / len(labels_tertile) for v in ["UP", "FLAT", "DOWN"]}

{
    "class_balance": {
        "binary_1day_pct_up": round(sum(labels_1d) / len(labels_1d), 3),
        "binary_5day_pct_up": round(sum(labels_5d) / len(labels_5d), 3),
        "binary_20day_pct_up": round(sum(labels_20d) / len(labels_20d), 3),
        "sharpe_5day_pct_1": round(sum(labels_sharpe) / len(labels_sharpe), 3),
        "tertile_5day": {k: round(v, 3) for k, v in tertile_counts.items()},
    },
    "autocorrelation_lag1": {
        "binary_1day": autocorr(labels_1d),
        "binary_5day": autocorr(labels_5d),
        "binary_20day": autocorr(labels_20d),
        "note": "higher autocorr = more predictable structure (longer horizon smooths noise)",
    },
    "insight": {
        "1day_binary": "~50/50, near-random — pure noise at daily frequency",
        "5day_binary": "slightly more structure due to weekly autocorrelation",
        "20day_binary": "more structure but feature/label windows may overlap (leakage risk if feature window >= 20)",
        "sharpe_5day": "fewer 'up' labels — filters out small moves below vol-adjusted threshold",
        "tertile": "creates more actionable signal by removing flat/noisy middle tier",
    },
}


### G. Train/Validation Strategies — Walk-Forward Is the Standard

For time-series, the validation strategy must respect the arrow of time.

**Three approaches in order of correctness:**

**1. Single time-based split**
Train on first 80%, test on last 20%. Fast. Problem: one evaluation period — you don't know if
performance is stable or just lucky for that specific regime.

**2. Walk-forward expanding window** (industry standard)
```
|------ train -------|--- test ---|
|---------- train --------|--- test ---|
|-------------- train ------------|--- test ---|
```
Gives N evaluation periods → distribution of performance and in-out gap estimates.

**3. Purging and embargoing**
If your feature window is W bars wide, the last W rows of each training fold share inputs with the
first rows of the test fold. **Purging** removes those overlapping rows from train.
**Embargoing** skips a buffer after the train boundary before starting the test period.
This is the rigorous approach for high-frequency or overlapping-window features.

**What to look for in walk-forward results:**
- High std of test accuracy across folds → regime-dependent signal (unreliable)
- Consistently large in-out gap → overfitting or structural leakage
- Declining test accuracy over time → concept drift (the market changed)


In [ ]:
import random
import statistics


random.seed(42)
prices_g = generate_gbm_prices(n=600)
returns_g = [(prices_g[i] - prices_g[i - 1]) / prices_g[i - 1] for i in range(1, len(prices_g))]

FEATURE_WINDOW = 20

rows_g = []
for i in range(FEATURE_WINDOW, len(returns_g) - 1):
    feat = statistics.mean(returns_g[i - FEATURE_WINDOW:i])
    label = 1 if returns_g[i] > 0 else 0
    rows_g.append((feat, label))

n = len(rows_g)


def train_and_eval(train_rows, test_rows):
    """Find best threshold on train; evaluate on test."""
    thresholds = sorted(set(f for f, _ in train_rows))
    best_t, best_tr_acc = thresholds[0], 0.0
    for t in thresholds:
        acc = sum((1 if f > t else 0) == l for f, l in train_rows) / len(train_rows)
        if acc > best_tr_acc:
            best_tr_acc, best_t = acc, t
    test_acc = sum((1 if f > best_t else 0) == l for f, l in test_rows) / len(test_rows)
    return best_tr_acc, test_acc


# --- Single time split ---
split = int(0.8 * n)
tr_single, te_single = train_and_eval(rows_g[:split], rows_g[split:])

# --- Walk-forward expanding window with embargo ---
MIN_TRAIN = 150
STEP = 40
EMBARGO = FEATURE_WINDOW  # skip window-size rows to prevent feature overlap

wf_results = []
t = MIN_TRAIN
while t + EMBARGO + STEP <= n:
    tr_rows = rows_g[:t]
    te_rows = rows_g[t + EMBARGO: t + EMBARGO + STEP]
    if len(te_rows) < 10:
        break
    tr_acc, te_acc = train_and_eval(tr_rows, te_rows)
    wf_results.append({
        "train_end": t,
        "embargo_end": t + EMBARGO,
        "test_end": t + EMBARGO + STEP,
        "train_acc": round(tr_acc, 3),
        "test_acc": round(te_acc, 3),
        "gap": round(tr_acc - te_acc, 3),
    })
    t += STEP

test_accs = [r["test_acc"] for r in wf_results]
gaps = [r["gap"] for r in wf_results]
majority_baseline = sum(l for _, l in rows_g[split:]) / len(rows_g[split:])

{
    "single_split": {
        "train_acc": round(tr_single, 3),
        "test_acc": round(te_single, 3),
        "in_out_gap": round(tr_single - te_single, 3),
    },
    "walk_forward": {
        "n_folds": len(wf_results),
        "test_acc_mean": round(statistics.mean(test_accs), 3),
        "test_acc_std": round(statistics.stdev(test_accs) if len(test_accs) > 1 else 0, 3),
        "gap_mean": round(statistics.mean(gaps), 3),
        "gap_std": round(statistics.stdev(gaps) if len(gaps) > 1 else 0, 3),
    },
    "majority_baseline": round(majority_baseline, 3),
    "fold_sample": wf_results[:3],
    "insight": {
        "high_gap_std": "performance varies by regime — signal is not stable",
        "declining_test_acc": "concept drift — the signal stops working as time progresses",
        "embargo_matters": f"skipping {EMBARGO} rows prevents {EMBARGO}-day window overlap leakage",
    },
}


### H. Feature Window Size — Noise vs. Lag Trade-Off

Every rolling feature has a window parameter. It is a hyperparameter with a measurable trade-off:

- **Short window (3–5 bars):** very responsive but mostly captures noise
- **Medium window (10–30 bars):** the typical "20-day" standard — balanced response
- **Long window (60–120 bars):** smooth but lagged — the signal appears after the opportunity

**Information Coefficient (IC):**
IC = correlation between a feature at time t and the forward label at t+1.
IC > 0 means the feature positively predicts the label.
IC near 0 means no relationship.
IC is how you empirically find the best window on your own data.

**EMA vs SMA:**
- SMA: all observations in the window equally weighted
- EMA: recent observations weighted more (decays geometrically with factor α = 2/(span+1))
- EMA at span N ≈ SMA at window N/2 in responsiveness

**Practical rule:** measure IC across windows on a hold-out period of your actual data.
There is no universal optimal window — it depends on the instrument, regime, and signal type.


In [ ]:
import statistics
import math


random.seed(42)
prices_h = generate_gbm_prices(n=600, drift=0.0004, vol=0.015)
returns_h = [(prices_h[i] - prices_h[i - 1]) / prices_h[i - 1] for i in range(1, len(prices_h))]
fwd_returns = returns_h[1:] + [None]  # forward 1-day return (label for IC)


def sma_feature(returns, window):
    result = [None] * window
    for i in range(window, len(returns)):
        result.append(statistics.mean(returns[i - window:i]))
    return result


def ema_feature(returns, span):
    alpha = 2.0 / (span + 1)
    ema = returns[0]
    result = []
    for r in returns:
        ema = alpha * r + (1 - alpha) * ema
        result.append(ema)
    return result


def information_coefficient(feature, label):
    """Pearson correlation between feature[t] and label[t]."""
    pairs = [(f, l) for f, l in zip(feature, label) if f is not None and l is not None]
    if len(pairs) < 20:
        return None
    fs, ls = zip(*pairs)
    n = len(fs)
    mu_f, mu_l = statistics.mean(fs), statistics.mean(ls)
    cov = sum((f - mu_f) * (l - mu_l) for f, l in zip(fs, ls)) / n
    std_f = math.sqrt(sum((f - mu_f) ** 2 for f in fs) / n)
    std_l = math.sqrt(sum((l - mu_l) ** 2 for l in ls) / n)
    return cov / (std_f * std_l) if std_f > 0 and std_l > 0 else 0.0


sma_windows = [3, 5, 10, 20, 40, 60]
ema_spans = [5, 10, 20, 40]

ic_sma = []
for w in sma_windows:
    feat = sma_feature(returns_h, w)
    ic = information_coefficient(feat, fwd_returns)
    feature_vals = [v for v in feat if v is not None]
    ic_sma.append({
        "type": "SMA",
        "window": w,
        "IC": round(ic, 5) if ic is not None else None,
        "feature_std": round(statistics.stdev(feature_vals), 6),
    })

ic_ema = []
for span in ema_spans:
    feat = ema_feature(returns_h, span)
    ic = information_coefficient(feat, fwd_returns)
    ic_ema.append({
        "type": "EMA",
        "span": span,
        "IC": round(ic, 5) if ic is not None else None,
        "feature_std": round(statistics.stdev(feat), 6),
    })

best_sma = max(ic_sma, key=lambda x: abs(x["IC"] or 0))
{
    "ic_by_sma_window": ic_sma,
    "ic_by_ema_span": ic_ema,
    "best_sma_window": best_sma,
    "insight": {
        "short_windows": "high feature std (responsive) but IC typically weak — mostly noise",
        "long_windows": "low feature std (smooth) but IC lags — signal arrives late",
        "ema_vs_sma": "EMA at span N roughly equals SMA at window N/2 in responsiveness",
        "key_rule": "always measure IC on your own instrument and time period — no universal answer",
        "note": "GBM has very weak autocorrelation by design; real markets vary significantly",
    },
}


### I. Class Imbalance and Evaluation Metrics

Directional prediction is close to 50/50 in efficient markets. Small imbalances and naive metrics
lead to false confidence.

**The accuracy trap:**
If 54% of days are "up", a classifier that always predicts "up" achieves 54% accuracy.
A model at 55% that predicts "up" every day has learned nothing — it's just the market drift.

**Metrics that matter:**

| Metric | Formula | Measures |
| --- | --- | --- |
| **Accuracy** | (TP + TN) / N | Overall correctness — misleading under imbalance |
| **Precision** | TP / (TP + FP) | Of all "up" predictions, how many were right? |
| **Recall** | TP / (TP + FN) | Of all actual "up" days, how many did we catch? |
| **F1** | 2·P·R / (P+R) | Harmonic mean — penalizes both failure modes equally |
| **Lift** | precision / base_rate − 1 | How much better than random? 0 = worthless |

**The economic perspective:**
In trading, what ultimately matters is not classification accuracy but the **profit and loss of acting on predictions**.
A model that's 55% accurate on large-move days is worth far more than one that's 60% accurate on small-move days.
Position-sizing by conviction (model confidence) matters more than any single metric.

**Threshold tuning:**
The default 0.5 decision threshold is arbitrary. Moving it:
- Higher threshold → higher precision, lower recall (fewer but more confident predictions)
- Lower threshold → higher recall, lower precision (more predictions, more noise)


In [ ]:
import random
import statistics
import math


random.seed(42)
prices_i = generate_gbm_prices(n=500, drift=0.0005, vol=0.015)
returns_i = [(prices_i[i] - prices_i[i - 1]) / prices_i[i - 1] for i in range(1, len(prices_i))]

# Binary labels
labels_i = [1 if r > 0 else 0 for r in returns_i]
base_rate = sum(labels_i) / len(labels_i)

# Feature: 10-day momentum
feat_10 = [None] * 10 + [statistics.mean(returns_i[i - 10:i]) for i in range(10, len(returns_i))]
feat_clean = [f if f is not None else 0.0 for f in feat_10]
feat_std = statistics.stdev(feat_clean)


def compute_metrics(y_true, y_pred, name):
    pairs = [(t, p) for t, p in zip(y_true, y_pred) if p is not None]
    if not pairs:
        return {"classifier": name, "n": 0}
    n = len(pairs)
    tp = sum(t == 1 and p == 1 for t, p in pairs)
    tn = sum(t == 0 and p == 0 for t, p in pairs)
    fp = sum(t == 0 and p == 1 for t, p in pairs)
    fn = sum(t == 1 and p == 0 for t, p in pairs)
    accuracy = (tp + tn) / n
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    lift = precision / base_rate - 1 if base_rate > 0 else 0.0
    return {
        "classifier": name,
        "coverage": round(n / len(y_true), 3),
        "accuracy": round(accuracy, 3),
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "lift_over_base": round(lift, 3),
    }


# Classifier 1: always predict majority
preds_majority = [1] * len(labels_i)

# Classifier 2: noisy but slightly better than random
random.seed(5)
preds_noisy = [
    l if random.random() < 0.57 else 1 - l
    for l in labels_i
]

# Classifier 3: high precision — only predict when signal is strong (top quartile)
feat_q75 = sorted(feat_clean)[int(0.75 * len(feat_clean))]
preds_highprec = [
    (1 if f > 0 else 0) if abs(f) >= feat_q75 else None
    for f in feat_clean
]

# Threshold analysis on noisy classifier
thresholds = [0.0, feat_std * 0.3, feat_std * 0.6, feat_std * 1.0, feat_std * 1.5]
threshold_analysis = []
for thr in thresholds:
    preds = [1 if f > thr else 0 for f in feat_clean]
    m = compute_metrics(labels_i, preds, f"threshold={thr:.4f}")
    threshold_analysis.append(m)

{
    "base_rate_pct_up": round(base_rate, 3),
    "classifier_comparison": [
        compute_metrics(labels_i, preds_majority, "always_majority"),
        compute_metrics(labels_i, preds_noisy, "noisy_57pct"),
        compute_metrics(labels_i, preds_highprec, "high_precision_25pct_coverage"),
    ],
    "threshold_tradeoff": threshold_analysis,
    "insight": {
        "accuracy_trap": "always_majority achieves high accuracy with zero lift — worthless",
        "precision_recall": "high-precision model makes fewer but better calls",
        "threshold_moves": "raising threshold: precision up, recall down, coverage down",
        "economic_metric": "lift * expected_return_per_correct_trade = practical value",
    },
}


### J. Principles Summary — Interview-Ready

**Price representation**
- Use returns or log-returns, not raw prices. Raw prices are non-stationary.
- Log-returns are additive across periods (exact); raw returns require compounding.

**Gap filling**
- Forward-fill prices (last known price is still valid).
- Zero-fill volume (no activity = zero activity).
- Never backward-fill anything that represents a measurement at a point in time.

**Outlier handling**
- Distinguish data errors (bad ticks) from real extreme events — don't remove real extremes.
- Winsorize features to cap magnitude without removing the data point entirely.
- Rolling Z-score clip is best for point anomalies; log-transform for right-skewed distributions.

**Normalization**
- Never normalize with global (full-sample) statistics before a time-based split — that's leakage.
- Always use rolling/walk-forward normalization: normalize at t using only data available up to t.

**Leakage — all five types**
- Random train/test split on time-series is wrong. Always split on time.
- Feature construction must use only data available at prediction time.
- Global scaling, future-bar prices, retroactively corrected data are all leakage sources.
- Include delisted instruments (survivorship bias is selection leakage).
- Use bitemporal/as-of data for any reference data that gets retroactively corrected.

**Label construction**
- Longer horizons reduce noise but increase window-overlap leakage risk.
- Volatility-adjusted (Sharpe-style) labels are more regime-consistent.
- Class balance matters — measure it before building a model.

**Validation**
- Walk-forward expanding window is the industry standard for time-series model evaluation.
- Measure and report the in-out performance gap across multiple folds.
- Purge and embargo rows near fold boundaries when features have long lookback windows.

**Metrics**
- Accuracy alone is misleading for near-balanced directional prediction.
- Lift over base rate (precision / base_rate − 1) is the most interpretable trading metric.
- The economic value (PnL of acting on predictions) is the ultimate measure.

**The meta-rule:** every data decision is a hypothesis about how the world works.
Make it explicit, measure the effect, and document why you chose it.


## 13. Mini Lab: End-to-End Trading Data Pipeline

This lab ties together every concept from this notebook into a single pipeline:

1. **Ingest** from 2 mock sources with different schemas
2. **Normalize** to canonical Trade model via symbol mapper
3. **Quality check** the normalized data
4. **Reconcile** across sources
5. **Compute point-in-time features** with explicit lookback windows
6. **Create a dataset snapshot** for reproducibility
7. **Track the experiment** with full traceability

This is the pipeline pattern you would build in production (with Kafka, TimescaleDB, and a feature store instead of in-memory lists).

In [ ]:
def run_pipeline() -> dict:
    """End-to-end trading data pipeline with full traceability."""

    pipeline_start = datetime.now(timezone.utc)
    print("=" * 70)
    print("  END-TO-END TRADING DATA PIPELINE")
    print("=" * 70)
    print()

    # --- Step 1: Ingest from mock sources ---
    print("STEP 1: Ingest from 2 mock vendors")
    # Generate richer mock data: 100 trades from each vendor
    random.seed(42)
    base_t = datetime(2025, 4, 4, 10, 0, 0, tzinfo=timezone.utc)
    alpha_raw = []
    beta_raw = []

    for i in range(100):
        ts = base_t + timedelta(seconds=i * 3)
        price = 1.08400 + random.gauss(0, 0.0005)
        size = random.choice([1_000_000, 2_000_000, 3_000_000, 5_000_000])
        side_char = random.choice(["B", "S"])

        alpha_raw.append({
            "id": f"A{i:04d}", "sym": "EUR/USD", "px": round(price, 5),
            "qty": size, "side": side_char, "ts": ts.timestamp(), "venue": "EBS",
        })

        # Beta gets 90% of the trades (simulates coverage gap)
        if random.random() < 0.9:
            beta_raw.append({
                "trade_ref": f"B-{10000+i}", "instrument": "EURUSD",
                "execution_price": f"{price:.5f}", "amount": str(size),
                "direction": "BUY" if side_char == "B" else "SELL",
                "timestamp": ts.isoformat(), "exchange": "EBS",
            })

    print(f"  Alpha: {len(alpha_raw)} raw records")
    print(f"  Beta:  {len(beta_raw)} raw records")
    print()

    # --- Step 2: Normalize to canonical model ---
    print("STEP 2: Normalize to canonical Trade model")
    alpha_normalized = [normalize_alpha(r) for r in alpha_raw]
    beta_normalized = [normalize_beta(r) for r in beta_raw]
    all_trades = alpha_normalized + beta_normalized
    print(f"  Alpha normalized: {len(alpha_normalized)} trades")
    print(f"  Beta normalized:  {len(beta_normalized)} trades")
    print(f"  Total:            {len(all_trades)} trades")
    print()

    # --- Step 3: Quality checks ---
    print("STEP 3: Run quality checks")
    alpha_report = run_trade_quality_checks(alpha_normalized)
    beta_report = run_trade_quality_checks(beta_normalized)
    print(f"  Alpha quality: {'PASS' if alpha_report.passed else 'FAIL'} ({alpha_report.summary})")
    print(f"  Beta quality:  {'PASS' if beta_report.passed else 'FAIL'} ({beta_report.summary})")

    if not alpha_report.passed or not beta_report.passed:
        print("  PIPELINE BLOCKED: Critical quality check failed!")
        return {"status": "BLOCKED"}
    print(f"  Quality gate: PASSED")
    print()

    # --- Step 4: Reconcile ---
    print("STEP 4: Reconcile across vendors")
    a_keys = {reconciliation_key(t): t for t in alpha_normalized}
    b_keys = {reconciliation_key(t): t for t in beta_normalized}
    matched = set(a_keys.keys()) & set(b_keys.keys())
    a_only = set(a_keys.keys()) - set(b_keys.keys())
    b_only = set(b_keys.keys()) - set(a_keys.keys())
    total_unique = len(matched) + len(a_only) + len(b_only)
    recon_rate = len(matched) / total_unique if total_unique > 0 else 0

    print(f"  Matched:          {len(matched)}")
    print(f"  Alpha-only:       {len(a_only)}")
    print(f"  Beta-only:        {len(b_only)}")
    print(f"  Reconciliation:   {recon_rate:.1%}")
    print()

    # --- Step 5: Deduplicate and build unified trade list ---
    print("STEP 5: Deduplicate into golden record set")
    # For matched trades, prefer Alpha (arrived first). Add unmatched from both.
    golden_trades = []
    for key in matched:
        golden_trades.append(a_keys[key])  # prefer Alpha
    for key in a_only:
        golden_trades.append(a_keys[key])
    for key in b_only:
        golden_trades.append(b_keys[key])
    golden_trades.sort(key=lambda t: t.event_time)
    print(f"  Golden record count: {len(golden_trades)}")
    print()

    # --- Step 6: Compute features ---
    print("STEP 6: Compute point-in-time features")
    # Build a simple price series from golden trades (aggregate to 1-trade "bars")
    price_series = [
        {
            "date": t.event_time,
            "price": t.price,
            "return_pct": 0.0,  # placeholder, will be overwritten
        }
        for t in golden_trades
    ]
    # Compute returns
    for i in range(1, len(price_series)):
        prev = float(price_series[i - 1]["price"])
        curr = float(price_series[i]["price"])
        price_series[i]["return_pct"] = round((curr - prev) / prev * 100, 6)

    features = compute_features(price_series, lookback=20)
    valid_features_pipeline = [f for f in features if f.get("volatility_20d") is not None]
    print(f"  Total observations: {len(features)}")
    print(f"  Valid features:     {len(valid_features_pipeline)} (after lookback warmup)")
    print()

    # --- Step 7: Create dataset snapshot ---
    print("STEP 7: Create dataset snapshot")
    pipeline_snapshot = create_snapshot(
        name="eurusd_pipeline_demo",
        feature_rows=valid_features_pipeline,
        feature_view_name="fx_momentum_features",
        feature_view_version="1.0.0",
        as_of_time=pipeline_start,
        source_versions={"vendor_alpha": "mock-v1", "vendor_beta": "mock-v1"},
        train_start=valid_features_pipeline[0]["date"] if valid_features_pipeline else None,
        train_end=valid_features_pipeline[-1]["date"] if valid_features_pipeline else None,
    )
    print(f"  snapshot_id:   {pipeline_snapshot.snapshot_id}")
    print(f"  rows:          {pipeline_snapshot.row_count}")
    print(f"  content_hash:  {pipeline_snapshot.content_hash[:16]}...")
    print()

    # --- Step 8: Track experiment ---
    print("STEP 8: Track experiment run")
    pipeline_end = datetime.now(timezone.utc)
    pipeline_run = ExperimentRun(
        run_id=f"pipeline-{uuid.uuid4().hex[:8]}",
        experiment_name="eurusd_pipeline_demo",
        dataset_snapshot_id=pipeline_snapshot.snapshot_id,
        feature_view_name="fx_momentum_features",
        feature_view_version="1.0.0",
        model_type="pipeline_only",
        parameters={"lookback": 20, "recon_rate_threshold": 0.8},
        metrics={
            "alpha_trade_count": len(alpha_normalized),
            "beta_trade_count": len(beta_normalized),
            "reconciliation_rate": round(recon_rate, 3),
            "golden_trade_count": len(golden_trades),
            "feature_count": len(valid_features_pipeline),
        },
        code_version="notebook-09-demo",
        run_start=pipeline_start,
        run_end=pipeline_end,
    )
    print(f"  run_id:        {pipeline_run.run_id}")
    print(f"  duration:      {pipeline_run.duration_seconds:.2f}s")
    print(f"  metrics:       {pipeline_run.metrics}")
    print()

    print("=" * 70)
    print("  PIPELINE COMPLETE")
    print("=" * 70)
    print()
    print("  Full traceability chain:")
    print(f"    Run:      {pipeline_run.run_id}")
    print(f"    Dataset:  {pipeline_snapshot.snapshot_id}")
    print(f"    Features: fx_momentum_features v1.0.0")
    print(f"    Sources:  {pipeline_snapshot.source_versions}")
    print(f"    Quality:  Alpha={'PASS' if alpha_report.passed else 'FAIL'}, Beta={'PASS' if beta_report.passed else 'FAIL'}")
    print(f"    Recon:    {recon_rate:.1%} match rate")

    return {
        "status": "SUCCESS",
        "run": pipeline_run,
        "snapshot": pipeline_snapshot,
        "recon_rate": recon_rate,
    }


# Run the full pipeline
result = run_pipeline()

## 14. Exit Criteria

Do not move on until you can answer these without searching.

**Market data entities:**
- What is the difference between event time, processing time, and as-of time?
- Why does point-in-time correctness matter for backtesting?
- What is a biternporal record and what problem does it solve?

**Feature engineering:**
- What is the absolute rule about feature construction in trading ML?
- Why must every feature use only a trailing window, never a centered or forward window?

**Data representation:**
- Why are raw prices bad features? What should you use instead?
- Why are log-returns additive but raw returns are not?
- What is the practical implication for multi-day return calculations?

**Gap filling:**
- What effect does forward-fill have on rolling volatility?
- Why is zero-fill appropriate for volume but not for price?
- When should you leave missing data as None instead of filling it?

**Outlier handling:**
- What is the difference between a data error and a real extreme event?
- Why is removing flash crash data from training dangerous?
- When would you use winsorization vs. rolling Z-score clipping?

**Normalization:**
- Why does global Z-score normalization introduce leakage in time-series?
- What is rolling normalization and why is it the safe alternative?
- What does GARCH clustering imply about normalization windows?

**Leakage — all five types:**
- What is split-time leakage and how does it inflate reported metrics?
- What is feature-time leakage and how do you fix it?
- What is survivorship bias and why is it selection leakage?
- What is point-in-time reference data and when do you need it?

**Label construction:**
- How does prediction horizon affect class balance and signal quality?
- What is a Sharpe-style (volatility-adjusted) label and when is it better?
- What is the window-overlap leakage risk with long-horizon labels?

**Validation:**
- What is walk-forward expanding window validation?
- What is purging and why is it needed when features have long lookback windows?
- What does a large in-sample vs. out-of-sample gap signal?

**Metrics:**
- Why is accuracy misleading for near-balanced directional prediction?
- What does lift over base rate measure?
- What does the economic (PnL) metric capture that accuracy does not?

**Dataset versioning:**
- Why do you need dataset snapshots for ML experiment reproducibility?
- What is a FeatureView and what problem does it solve?


## Interview Question Bank

Answer out loud, not just in writing. Each question is a realistic interview scenario.

**Domain and data modelling:**
- You receive trade data from two venues with different timestamp formats and different timezone conventions. Walk me through how you normalize them for a unified time-series.
- A data vendor backfills corrected closing prices for the past six months. How does this affect your existing trained models?
- How would you design a schema for storing instrument reference data that supports SCD Type 2 history and point-in-time queries?

**Feature engineering:**
- A colleague's notebook uses `sklearn.preprocessing.StandardScaler.fit_transform(X_all)` on the full feature matrix before splitting. What is wrong with this?
- You have a feature computed with a 60-day rolling window. At what point in time can your model first make a valid prediction?
- Walk me through how you would compute a rolling VWAP using only past data. Where could a centered moving average introduce leakage?

**Data decisions:**
- Your training set has 7% missing bars in a particular symbol. Walk me through three gap-filling strategies and their downstream effects on rolling-vol features.
- A flash crash on a single day generates a −20% return in your return series. How do you decide whether to clean it or keep it?
- You're normalizing features for a cross-sectional model. Why is rank normalization sometimes preferable to Z-score normalization?

**Model evaluation:**
- Your model achieves 58% accuracy on the training set and 51% accuracy on the test set. What are three possible explanations?
- Explain walk-forward validation to someone who has only heard of train/test split. Why does it matter for trading models?
- What is the difference between purging and embargoing in walk-forward validation?

**Metrics:**
- Your binary classifier predicts "up" 30% of the time with 70% precision, while the market goes up 53% of the time. Is this a good model? What metrics would you look at?
- Why might a model with 55% accuracy still lose money in production?

**System design:**
- You need to build a pipeline that produces a daily feature dataset for 500 instruments, ensuring all features are point-in-time correct and the dataset is reproducible. Walk me through the key design decisions.
- How would you detect when a production model's distribution of predictions starts diverging from its training-time distribution?
